In [27]:
import os
import sys
import time
import random
import platform

from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from transformers import (
    AutoTokenizer,
    AutoModel,
)

In [28]:
# =========================
# 网络代理
# =========================

PROXY_URL = (
    "http://127.0.0.1:23897"
)

os.environ["HTTP_PROXY"] = PROXY_URL
os.environ["HTTPS_PROXY"] = PROXY_URL
os.environ["http_proxy"] = PROXY_URL
os.environ["https_proxy"] = PROXY_URL

os.environ["HF_ENDPOINT"] = (
    "https://huggingface.co"
)

os.environ[
    "HF_HUB_DOWNLOAD_TIMEOUT"
] = "300"

os.environ[
    "HF_HUB_ETAG_TIMEOUT"
] = "300"

os.environ[
    "TOKENIZERS_PARALLELISM"
] = "false"

In [29]:
# =========================
# 路径
# =========================

PROJECT_ROOT = Path(
    "/workspace/E-DAIC"
)

DATA_DIR = (
    PROJECT_ROOT
    / "original_data"
)

SPLIT_DIR = (
    PROJECT_ROOT
    / "splits"
)

LABEL_DIR = (
    PROJECT_ROOT
    / "others"
)

TRAIN_SPLIT_PATH = (
    SPLIT_DIR
    / "train_split.csv"
)

VAL_SPLIT_PATH = (
    SPLIT_DIR
    / "dev_split.csv"
)

DETAILED_LABEL_PATH = (
    LABEL_DIR
    / "Detailed_PHQ8_Labels.csv"
)


# RoBERTa句向量磁盘缓存
EMBEDDING_CACHE_DIR = (
    PROJECT_ROOT
    / "cache"
    / "phq8_text_embeddings"
)


# 新实验Checkpoint目录
CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "checkpoints"
    / "phq8_text_paperlike"
)


EMBEDDING_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

In [30]:
# =========================
# 8道PHQ-8题目
# =========================

PHQ8_QUESTION_COLUMNS = [
    "PHQ_8NoInterest",
    "PHQ_8Depressed",
    "PHQ_8Sleep",
    "PHQ_8Tired",
    "PHQ_8Appetite",
    "PHQ_8Failure",
    "PHQ_8Concentrating",
    "PHQ_8Moving",
]


# =========================
# 模型配置
# =========================

MODEL_NAME = (
    "sentence-transformers/"
    "all-distilroberta-v1"
)

MAX_TURNS = 120
EMBEDDING_DIM = 768

LSTM_HIDDEN_SIZE = 50
NUM_ATTENTION_HEADS = 4
NUM_CLASSES = 4


# =========================
# 训练配置
# =========================

BATCH_SIZE = 10
NUM_EPOCHS = 20

LEARNING_RATE = 5e-4
OPTIMIZER_EPS = 1e-8
WEIGHT_DECAY = 1e-3

MAX_GRAD_NORM = 1.0

ALPHA = 1.0
BETA = 0.5

EPS = 1e-12

EXPERIMENT_SEEDS = [
    42,
    100,
    1234,
]

In [38]:
def set_seed(seed):
    print(
        f"设置随机种子：{seed}"
    )

    # Python内置随机模块
    random.seed(seed)

    # NumPy随机数
    np.random.seed(seed)

    # PyTorch的CPU随机数
    torch.manual_seed(seed)

    # 所有可见GPU的随机数
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    # 尽量让cuDNN每次选择相同算法
    if hasattr(
        torch.backends,
        "cudnn",
    ):
        torch.backends.cudnn.deterministic = (
            True
        )

        torch.backends.cudnn.benchmark = (
            False
        )


set_seed(
    EXPERIMENT_SEEDS[0]
)

train_split_df = pd.read_csv(
    TRAIN_SPLIT_PATH
)

val_split_df = pd.read_csv(
    VAL_SPLIT_PATH
)

detailed_labels_df = pd.read_csv(
    DETAILED_LABEL_PATH
)
required_label_columns = [
    "Participant_ID",
    *PHQ8_QUESTION_COLUMNS,
]


missing_label_columns = (
    set(required_label_columns)
    - set(detailed_labels_df.columns)
)


if missing_label_columns:
    raise ValueError(
        "详细标签文件缺少这些列："
        f"{missing_label_columns}"
    )


print("8道题标签列检查通过")
label_subset_df = (
    detailed_labels_df[
        required_label_columns
    ]
    .copy()
)


train_metadata_df = (
    train_split_df.merge(
        label_subset_df,
        on="Participant_ID",
        how="left",
        validate="one_to_one",
    )
)


val_metadata_df = (
    val_split_df.merge(
        label_subset_df,
        on="Participant_ID",
        how="left",
        validate="one_to_one",
    )
)
for split_name, metadata_df in [
    (
        "train",
        train_metadata_df,
    ),
    (
        "val",
        val_metadata_df,
    ),
]:
    missing_count = (
        metadata_df[
            PHQ8_QUESTION_COLUMNS
        ]
        .isna()
        .sum()
        .sum()
    )

    if missing_count > 0:
        raise ValueError(
            f"{split_name}存在"
            f"{missing_count}个空标签"
        )

    label_values = set(
        metadata_df[
            PHQ8_QUESTION_COLUMNS
        ]
        .to_numpy()
        .reshape(-1)
        .tolist()
    )

    if not label_values.issubset(
        {0, 1, 2, 3}
    ):
        raise ValueError(
            f"{split_name}出现非法标签："
            f"{label_values}"
        )

    metadata_df[
        PHQ8_QUESTION_COLUMNS
    ] = metadata_df[
        PHQ8_QUESTION_COLUMNS
    ].astype("int64")


train_participant_ids = set(
    train_metadata_df[
        "Participant_ID"
    ].tolist()
)

val_participant_ids = set(
    val_metadata_df[
        "Participant_ID"
    ].tolist()
)


overlap_ids = (
    train_participant_ids
    & val_participant_ids
)


if overlap_ids:
    raise ValueError(
        "训练集与验证集出现重叠："
        f"{sorted(overlap_ids)}"
    )


print(
    "训练参与者数量：",
    len(train_metadata_df),
)

print(
    "验证参与者数量：",
    len(val_metadata_df),
)

print(
    "训练/验证参与者没有重叠"
)

print(
    "全部标签取值：",
    sorted(
        set(
            train_metadata_df[
                PHQ8_QUESTION_COLUMNS
            ]
            .to_numpy()
            .reshape(-1)
            .tolist()
        )
    ),
)
device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

设置随机种子：42
8道题标签列检查通过
训练参与者数量： 163
验证参与者数量： 55
训练/验证参与者没有重叠
全部标签取值： [0, 1, 2, 3]


In [39]:
def make_class_count_table(
    metadata_df,
):
    count_dictionary = {}

    for question_column in (
        PHQ8_QUESTION_COLUMNS
    ):
        counts = (
            metadata_df[
                question_column
            ]
            .value_counts()
            .reindex(
                [0, 1, 2, 3],
                fill_value=0,
            )
        )

        count_dictionary[
            question_column
        ] = counts

    count_table = pd.DataFrame(
        count_dictionary
    )

    count_table.index.name = "Score"

    return count_table
train_class_count_df = (
    make_class_count_table(
        train_metadata_df
    )
)

val_class_count_df = (
    make_class_count_table(
        val_metadata_df
    )
)


print("训练集类别分布：")
display(train_class_count_df)

print("验证集类别分布：")
display(val_class_count_df)

训练集类别分布：


,PHQ_8NoInterest,PHQ_8Depressed,PHQ_8Sleep,PHQ_8Tired,PHQ_8Appetite,PHQ_8Failure,PHQ_8Concentrating,PHQ_8Moving
Score,,,,,,,,
0,81,68,69,50,77,76,93,128
1,57,63,40,65,42,44,37,20
2,20,18,27,27,24,24,14,9
3,5,14,27,21,20,19,19,6


验证集类别分布：


,PHQ_8NoInterest,PHQ_8Depressed,PHQ_8Sleep,PHQ_8Tired,PHQ_8Appetite,PHQ_8Failure,PHQ_8Concentrating,PHQ_8Moving
Score,,,,,,,,
0,25,25,22,16,19,25,31,39
1,22,19,17,23,22,17,13,12
2,4,7,8,9,10,8,8,3
3,4,4,8,7,4,5,3,1


In [40]:
MODEL_REVISION = (
    "842eaed40bee4d61673a81c92d5689a8fed7a09f"
)
print("开始加载Tokenizer和RoBERTa")


try:
    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_NAME,
            revision=MODEL_REVISION,
            local_files_only=True,
        )
    )

    embedder = (
        AutoModel.from_pretrained(
            MODEL_NAME,
            revision=MODEL_REVISION,
            local_files_only=True,
        )
    )

    print("从本地缓存加载成功")

except OSError:
    print(
        "本地缓存不完整，"
        "开始通过网络补充下载"
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_NAME,
            revision=MODEL_REVISION,
        )
    )

    embedder = (
        AutoModel.from_pretrained(
            MODEL_NAME,
            revision=MODEL_REVISION,
        )
    )

开始加载Tokenizer和RoBERTa


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/all-distilroberta-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


从本地缓存加载成功


In [41]:
embedder=embedder.to(device)

for parameter in embedder.parameters():
    parameter.requires_grad=False
embedder.eval()

roberta_hidden_size=embedder.config.hidden_size
roberta_total_parameters = sum(parameter.numel() for parameter in embedder.parameters())
roberta_trainable_parameters = sum(parameter.numel for parameter in embedder.parameters() if parameter.requires_grad)
print(
    "Tokenizer类型：",
    type(tokenizer),
)

print(
    "RoBERTa类型：",
    type(embedder),
)

print(
    "RoBERTa隐藏维度：",
    roberta_hidden_size,
)

print(
    "配置的句向量维度：",
    EMBEDDING_DIM,
)

print(
    "模型设备：",
    next(
        embedder.parameters()
    ).device,
)

print(
    "RoBERTa总参数量：",
    f"{roberta_total_parameters:,}",
)

print(
    "RoBERTa可训练参数量：",
    roberta_trainable_parameters,
)


if roberta_hidden_size != EMBEDDING_DIM:
    raise ValueError(
        "RoBERTa输出维度与"
        "EMBEDDING_DIM不一致"
    )

Tokenizer类型： <class 'transformers.models.roberta.tokenization_roberta.RobertaTokenizer'>
RoBERTa类型： <class 'transformers.models.roberta.modeling_roberta.RobertaModel'>
RoBERTa隐藏维度： 768
配置的句向量维度： 768
模型设备： cuda:0
RoBERTa总参数量： 82,118,400
RoBERTa可训练参数量： 0


In [42]:
def mean_pooling(model_output,attention_mask):
    token_embeddings=model_output.last_hidden_state

    expanded_mask=attention_mask.unsqueeze(-1).expand_as(token_embeddings).float()

    embedding_sum=torch.sum(token_embeddings*expanded_mask,dim=1)

    valid_token_count=expanded_mask.sum(dim=1).clamp(min=1e-9)

    sentence_embeddings=embedding_sum/valid_token_count

    return sentence_embeddings



In [43]:
CACHE_VERSION = (
    "distilroberta_"
    f"{MODEL_REVISION[:8]}_"
    f"turns{MAX_TURNS}_v1"
)
ACTIVE_CACHE_DIR = (
    EMBEDDING_CACHE_DIR
    / CACHE_VERSION
)
ACTIVE_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
print(
    "当前缓存版本：",
    CACHE_VERSION,
)

print(
    "当前缓存目录：",
    ACTIVE_CACHE_DIR,
)

当前缓存版本： distilroberta_842eaed4_turns120_v1
当前缓存目录： /workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1


In [44]:
def get_transcript_path(
    participant_id,
):
    participant_id = int(
        participant_id
    )

    transcript_path = (
        DATA_DIR
        / f"{participant_id}_P"
        / (
            f"{participant_id}_"
            "Transcript.csv"
        )
    )

    return transcript_path

In [45]:
def compute_participant_embeddings(participant_id):
    participant_id = int(participant_id)

    transcript_path = get_transcript_path(participant_id)

    if not transcript_path.exists():
        raise FileNotFoundError(f"找不到转录文件：{transcript_path}")

    transcript_df = pd.read_csv(transcript_path)

    if "Text" not in transcript_df.columns:
        raise ValueError(f"{transcript_path}中没有Text列")

    # 删除空值，转成字符串并清除两端空格
    text_series = transcript_df["Text"].dropna().astype(str).str.strip()


    # 删除清理后长度为0的文本
    text_list = [text for text in text_series.tolist() if len(text) > 0]


    # 最多保留前120轮
    text_list = text_list[:MAX_TURNS]

    real_turn_count = len(text_list)

    if real_turn_count == 0:
        raise ValueError(f"参与者{participant_id}没有有效文本")

    encoded_input = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")

    # BatchEncoding内部包括input_ids和attention_mask
    encoded_input = encoded_input.to(device)

    with torch.no_grad():
        model_output = embedder(**encoded_input)

    sentence_embeddings = mean_pooling(model_output=model_output, attention_mask=encoded_input["attention_mask"])

    # 建立固定的[120,768]全零矩阵
    padded_embeddings = torch.zeros((MAX_TURNS, EMBEDDING_DIM), dtype=sentence_embeddings.dtype, device=device)

    # 把真实句向量放在前面
    padded_embeddings[:real_turn_count] = sentence_embeddings

    # 每句话分别做L2归一化
    padded_embeddings = F.normalize(padded_embeddings, p=2, dim=1)

    # True表示padding，False表示真实对话
    key_padding_mask = torch.ones(MAX_TURNS, dtype=torch.bool, device=device)

    key_padding_mask[:real_turn_count] = False

    # 缓存统一保存在CPU
    padded_embeddings = padded_embeddings.detach().cpu().float()

    key_padding_mask = key_padding_mask.cpu()

    return padded_embeddings, key_padding_mask, real_turn_count
compute_participant_embeddings(participant_id=302)

(tensor([[ 0.0066, -0.0215,  0.0618,  ..., -0.0309,  0.0024,  0.0054],
         [ 0.0014,  0.0007,  0.0269,  ...,  0.0225,  0.0519, -0.0391],
         [ 0.0068,  0.0188,  0.0101,  ..., -0.0212, -0.0217,  0.0652],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]),
 tensor([False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
     

In [46]:
def get_embedding_cache_path(
    participant_id,
):
    participant_id = int(
        participant_id
    )

    return (
        ACTIVE_CACHE_DIR
        / f"{participant_id}.pt"
    )
def get_participant_embeddings(participant_id):
    participant_id=int(participant_id)

    transcript_path=get_transcript_path(participant_id)

    cache_path=get_embedding_cache_path(participant_id)

    if not transcript_path.exists():
        raise FileNotFoundError(
            f"找不到转录文件："
            f"{transcript_path}"
        )

    source_stat=transcript_path.stat()
    source_size=source_stat.st_size
    source_mtime_ns=source_stat.st_mtime_ns

    if cache_path.exists():
        cached_data = torch.load(
            cache_path,
            map_location="cpu",
            weights_only=True,
        )

        cache_is_valid = (
            cached_data.get(
                "participant_id"
            ) == participant_id

            and cached_data.get(
                "model_revision"
            ) == MODEL_REVISION

            and cached_data.get(
                "max_turns"
            ) == MAX_TURNS

            and cached_data.get(
                "embedding_dim"
            ) == EMBEDDING_DIM

            and cached_data.get(
                "source_size"
            ) == source_size

            and cached_data.get(
                "source_mtime_ns"
            ) == source_mtime_ns
        )
        if cache_is_valid:
            embeddings = cached_data[
                "embeddings"
            ]

            key_padding_mask = (
                cached_data[
                    "key_padding_mask"
                ]
            )

            real_turn_count = int(
                cached_data[
                    "real_turn_count"
                ]
            )

            if embeddings.shape != (
                MAX_TURNS,
                EMBEDDING_DIM,
            ):
                raise ValueError(
                    "缓存句向量形状错误："
                    f"{embeddings.shape}"
                )

            if key_padding_mask.shape != (
                MAX_TURNS,
            ):
                raise ValueError(
                    "缓存Mask形状错误："
                    f"{key_padding_mask.shape}"
                )

            return (
                embeddings,
                key_padding_mask,
                real_turn_count,
                True,
            )
    (
        embeddings,
        key_padding_mask,
        real_turn_count,
    ) = compute_participant_embeddings(
        participant_id
    )

    cached_data = {
        "participant_id": (
            participant_id
        ),
        "model_name": MODEL_NAME,
        "model_revision": (
            MODEL_REVISION
        ),
        "max_turns": MAX_TURNS,
        "embedding_dim": (
            EMBEDDING_DIM
        ),
        "source_size": source_size,
        "source_mtime_ns": (
            source_mtime_ns
        ),
        "real_turn_count": (
            real_turn_count
        ),
        "embeddings": embeddings,
        "key_padding_mask": (
            key_padding_mask
        ),
    }

    torch.save(
        cached_data,
        cache_path,
    )

    return (
        embeddings,
        key_padding_mask,
        real_turn_count,
        False,
    )

In [48]:
(
    embeddings_1,
    mask_1,
    real_turn_count_1,
    cache_hit_1,
) = get_participant_embeddings(
    302
)


print(
    "第一次是否命中缓存：",
    cache_hit_1,
)

print(
    "句向量形状：",
    embeddings_1.shape,
)

print(
    "Mask形状：",
    mask_1.shape,
)

print(
    "真实对话轮数：",
    real_turn_count_1,
)

print(
    "Padding轮数：",
    mask_1.sum().item(),
)

print(
    "句向量设备：",
    embeddings_1.device,
)

print(
    "Mask数据类型：",
    mask_1.dtype,
)

第一次是否命中缓存： True
句向量形状： torch.Size([120, 768])
Mask形状： torch.Size([120])
真实对话轮数： 99
Padding轮数： 21
句向量设备： cpu
Mask数据类型： torch.bool


In [53]:
def build_split_embedding_cache(metadata_df,split_name):
    participant_ids=metadata_df["Participant_ID"].astype(int).tolist()

    total_count=len(participant_ids)

    cache_hit_count=0
    cache_miss_count=0
    total_real_turns=0

    start_time=time.time()
    print(
        f"开始处理{split_name}："
        f"共{total_count}位参与者"
    )
    for index,participant_id in enumerate(participant_ids,start=1):
        embeddings,key_padding_mask,real_turn_count,cache_hit=get_participant_embeddings(participant_id)

        if embeddings.shape!=(MAX_TURNS,EMBEDDING_DIM):
            raise ValueError(
                f"参与者{participant_id}"
                f"句向量形状错误："
                f"{embeddings.shape}"
            )
        if key_padding_mask.shape!=(MAX_TURNS,):
            raise ValueError(
                f"参与者{participant_id}"
                f"Mask形状错误："
                f"{key_padding_mask.shape}"
            )
        if cache_hit:
            cache_hit_count+=1
        else:
            cache_miss_count+=1
        total_real_turns+=real_turn_count
        print(
            f"\r{split_name}进度："
            f"{index}/{total_count} "
            f"命中={cache_hit_count} "
            f"新计算={cache_miss_count}",
            end="",
        )
    elapsed_seconds = (
            time.time() - start_time
        )
    average_real_turns = (
        total_real_turns
        / total_count
    )

    result = {
        "split": split_name,
        "total": total_count,
        "cache_hits": (
            cache_hit_count
        ),
        "cache_misses": (
            cache_miss_count
        ),
        "average_real_turns": (
            average_real_turns
        ),
        "elapsed_seconds": (
            elapsed_seconds
        ),
    }

    return result

In [54]:
train_cache_result = (
    build_split_embedding_cache(
        metadata_df=(
            train_metadata_df
        ),
        split_name="train",
    )
)


val_cache_result = (
    build_split_embedding_cache(
        metadata_df=(
            val_metadata_df
        ),
        split_name="val",
    )
)

开始处理train：共163位参与者
train进度：163/163 命中=1 新计算=162开始处理val：共55位参与者
val进度：55/55 命中=0 新计算=55

In [68]:
print()
print("训练集缓存结果：")

for key, value in (
    train_cache_result.items()
):
    print(
        f"  {key}: {value}"
    )


print()
print("验证集缓存结果：")

for key, value in (
    val_cache_result.items()
):
    print(
        f"  {key}: {value}"
    )


训练集缓存结果：
  split: train
  total: 163
  cache_hits: 1
  cache_misses: 162
  average_real_turns: 88.38036809815951
  elapsed_seconds: 7.694563627243042

验证集缓存结果：
  split: val
  total: 55
  cache_hits: 0
  cache_misses: 55
  average_real_turns: 92.63636363636364
  elapsed_seconds: 2.16180419921875


In [69]:
all_required_participant_ids = set(
    train_metadata_df[
        "Participant_ID"
    ]
    .astype(int)
    .tolist()
) | set(
    val_metadata_df[
        "Participant_ID"
    ]
    .astype(int)
    .tolist()
)


cached_participant_ids = set()


for cache_path in (
    ACTIVE_CACHE_DIR.glob(
        "*.pt"
    )
):
    try:
        participant_id = int(
            cache_path.stem
        )

        cached_participant_ids.add(
            participant_id
        )

    except ValueError:
        # 忽略文件名不是参与者编号的pt文件
        pass


missing_cache_ids = (
    all_required_participant_ids
    - cached_participant_ids
)


print(
    "需要缓存的参与者数：",
    len(
        all_required_participant_ids
    ),
)

print(
    "当前缓存文件数：",
    len(
        cached_participant_ids
    ),
)

print(
    "缺少缓存的参与者：",
    sorted(
        missing_cache_ids
    ),
)


if missing_cache_ids:
    raise RuntimeError(
        "磁盘缓存尚不完整"
    )


print("全部参与者缓存建立完成")

需要缓存的参与者数： 218
当前缓存文件数： 218
缺少缓存的参与者： []
全部参与者缓存建立完成


In [70]:
class CachedPHQ8Dataset(Dataset):

    def __init__(self, metadata_df, split_name):
        super().__init__()

        self.split_name = split_name

        self.metadata_df = metadata_df.reset_index(drop=True).copy()

        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).to_numpy()

        # [参与者数量, 8]
        self.labels = torch.tensor(self.metadata_df[PHQ8_QUESTION_COLUMNS].to_numpy(), dtype=torch.long)

        embedding_list = []
        mask_list = []

        cache_hit_count = 0

        print(f"把{split_name}缓存载入内存")

        for index, participant_id in enumerate(self.participant_ids, start=1):
            embeddings, key_padding_mask, real_turn_count, cache_hit = get_participant_embeddings(participant_id)

            embedding_list.append(embeddings)

            mask_list.append(key_padding_mask)

            if cache_hit:
                cache_hit_count += 1

            print(f"\r{split_name}载入进度：{index}/{len(self.participant_ids)}", end="")

        print()

        # 163个[120,768]堆叠为[163,120,768]
        self.embeddings = torch.stack(embedding_list, dim=0)

        # 163个[120]堆叠为[163,120]
        self.key_padding_masks = torch.stack(mask_list, dim=0)

        if len(self.embeddings) != len(self.labels):
            raise ValueError(f"{split_name}向量数量与标签数量不一致")

        print(f"{split_name}载入完成，缓存命中：{cache_hit_count}/{len(self.participant_ids)}")

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        embeddings = self.embeddings[index]

        key_padding_mask = self.key_padding_masks[index]

        eight_question_labels = self.labels[index]

        return embeddings, key_padding_mask, eight_question_labels

In [71]:
train_dataset = (
    CachedPHQ8Dataset(
        metadata_df=(
            train_metadata_df
        ),
        split_name="train",
    )
)


val_dataset = (
    CachedPHQ8Dataset(
        metadata_df=(
            val_metadata_df
        ),
        split_name="val",
    )
)

把train缓存载入内存
train载入进度：163/163
train载入完成，缓存命中：163/163
把val缓存载入内存
val载入进度：55/55
val载入完成，缓存命中：55/55


In [72]:
print()
print(
    "训练句向量：",
    train_dataset.embeddings.shape,
)

print(
    "训练Mask：",
    train_dataset
    .key_padding_masks.shape,
)

print(
    "训练标签：",
    train_dataset.labels.shape,
)


print()
print(
    "验证句向量：",
    val_dataset.embeddings.shape,
)

print(
    "验证Mask：",
    val_dataset
    .key_padding_masks.shape,
)

print(
    "验证标签：",
    val_dataset.labels.shape,
)


训练句向量： torch.Size([163, 120, 768])
训练Mask： torch.Size([163, 120])
训练标签： torch.Size([163, 8])

验证句向量： torch.Size([55, 120, 768])
验证Mask： torch.Size([55, 120])
验证标签： torch.Size([55, 8])


In [73]:
def tensor_size_mb(
    tensor,
):
    byte_count = (
        tensor.numel()
        * tensor.element_size()
    )

    megabyte_count = (
        byte_count
        / 1024**2
    )


    return megabyte_count

In [74]:
train_memory_mb = (
    tensor_size_mb(
        train_dataset.embeddings
    )
    + tensor_size_mb(
        train_dataset
        .key_padding_masks
    )
    + tensor_size_mb(
        train_dataset.labels
    )
)


val_memory_mb = (
    tensor_size_mb(
        val_dataset.embeddings
    )
    + tensor_size_mb(
        val_dataset
        .key_padding_masks
    )
    + tensor_size_mb(
        val_dataset.labels
    )
)


print(
    "训练集内存占用：",
    round(
        train_memory_mb,
        2,
    ),
    "MB",
)

print(
    "验证集内存占用：",
    round(
        val_memory_mb,
        2,
    ),
    "MB",
)

print(
    "合计内存占用：",
    round(
        train_memory_mb
        + val_memory_mb,
        2,
    ),
    "MB",
)

训练集内存占用： 57.33 MB
验证集内存占用： 19.35 MB
合计内存占用： 76.68 MB


In [75]:
inspection_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


(
    batch_embeddings,
    batch_masks,
    batch_all_labels,
) = next(
    iter(inspection_loader)
)


print(
    "批次句向量：",
    batch_embeddings.shape,
)

print(
    "批次Mask：",
    batch_masks.shape,
)

print(
    "批次全部标签：",
    batch_all_labels.shape,
)


question_index = 0

question_labels = (
    batch_all_labels[
        :,
        question_index,
    ]
)


print(
    "当前题目：",
    PHQ8_QUESTION_COLUMNS[
        question_index
    ],
)

print(
    "取出后的单题标签：",
    question_labels.shape,
)

print(
    "当前Batch的Q1标签：",
    question_labels,
)

批次句向量： torch.Size([10, 120, 768])
批次Mask： torch.Size([10, 120])
批次全部标签： torch.Size([10, 8])
当前题目： PHQ_8NoInterest
取出后的单题标签： torch.Size([10])
当前Batch的Q1标签： tensor([1, 0, 0, 0, 0, 2, 2, 1, 3, 0])


In [76]:
class LSTMAttentionClassifier(
    nn.Module
):

    def __init__(self):
        super().__init__()

        # 双向LSTM
        self.lstm = nn.LSTM(
            input_size=(
                EMBEDDING_DIM
            ),
            hidden_size=(
                LSTM_HIDDEN_SIZE
            ),
            batch_first=True,
            bidirectional=True,
        )

        # 双向LSTM输出维度：
        # 50个正向 + 50个反向 = 100
        lstm_output_size = (
            LSTM_HIDDEN_SIZE * 2
        )

        # 多头自注意力
        self.attention = (
            nn.MultiheadAttention(
                embed_dim=(
                    lstm_output_size
                ),
                num_heads=(
                    NUM_ATTENTION_HEADS
                ),
                dropout=0.5,
                batch_first=True,
            )
        )

        # 分类头
        self.mlp = nn.Sequential(
            nn.Flatten(
                start_dim=1
            ),

            nn.Dropout(0.5),

            nn.Linear(
                MAX_TURNS
                * lstm_output_size,
                256,
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                256,
                NUM_CLASSES,
            ),
        )

    def forward(
        self,
        embeddings,
        key_padding_mask,
    ):
        # embeddings：
        # [batch, 120, 768]

        if embeddings.ndim != 3:
            raise ValueError(
                "embeddings必须是三维张量"
            )

        if key_padding_mask.ndim != 2:
            raise ValueError(
                "Mask必须是二维张量"
            )

        if embeddings.shape[:2] != (
            key_padding_mask.shape
        ):
            raise ValueError(
                "句向量前两维与Mask不一致"
            )

        # LSTM输出：
        # [batch, 120, 100]
        lstm_output, _ = self.lstm(
            embeddings
        )

        # 自注意力中：
        # query、key、value都使用LSTM输出
        attention_output, _ = (
            self.attention(
                query=lstm_output,
                key=lstm_output,
                value=lstm_output,
                key_padding_mask=(
                    key_padding_mask
                ),
                need_weights=False,
            )
        )

        # attention_output：
        # [batch, 120, 100]
        #
        # Flatten后：
        # [batch, 12000]
        #
        # MLP最终输出：
        # [batch, 4]
        logits = self.mlp(
            attention_output
        )

        return logits

In [77]:
set_seed(42)


inspection_model = (
    LSTMAttentionClassifier()
    .to(device)
)
inspection_embeddings = (
    batch_embeddings.to(device)
)

inspection_masks = (
    batch_masks.to(device)
)
inspection_model.eval()


with torch.no_grad():
    inspection_logits = (
        inspection_model(
            inspection_embeddings,
            inspection_masks,
        )
    )


print(
    "模型输入形状：",
    inspection_embeddings.shape,
)

print(
    "模型输出形状：",
    inspection_logits.shape,
)

print(
    "第一个样本logits：",
    inspection_logits[0],
)

print(
    "第一个样本概率：",
    torch.softmax(
        inspection_logits[0],
        dim=0,
    ),
)

print(
    "概率之和：",
    torch.softmax(
        inspection_logits[0],
        dim=0,
    ).sum().item(),
)

设置随机种子：42
模型输入形状： torch.Size([10, 120, 768])
模型输出形状： torch.Size([10, 4])
第一个样本logits： tensor([-0.0492,  0.0274,  0.0043,  0.0174], device='cuda:0')
第一个样本概率： tensor([0.2379, 0.2568, 0.2510, 0.2543], device='cuda:0')
概率之和： 1.0


In [78]:
def count_parameters(
    module,
):
    return sum(
        parameter.numel()
        for parameter in (
            module.parameters()
        )
        if parameter.requires_grad
    )
lstm_parameter_count = (
    count_parameters(
        inspection_model.lstm
    )
)

attention_parameter_count = (
    count_parameters(
        inspection_model.attention
    )
)

mlp_parameter_count = (
    count_parameters(
        inspection_model.mlp
    )
)

total_parameter_count = (
    count_parameters(
        inspection_model
    )
)


print(
    "LSTM参数：",
    f"{lstm_parameter_count:,}",
)

print(
    "Attention参数：",
    f"{attention_parameter_count:,}",
)

print(
    "MLP参数：",
    f"{mlp_parameter_count:,}",
)

print(
    "总可训练参数：",
    f"{total_parameter_count:,}",
)

LSTM参数： 328,000
Attention参数： 40,400
MLP参数： 3,073,284
总可训练参数： 3,441,684


In [79]:
del inspection_model
del inspection_embeddings
del inspection_masks
del inspection_logits


if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("检查模型已释放")

检查模型已释放


In [80]:
def compute_question_class_weights(
    labels_matrix,
    question_index,
    beta,
):
    # 从[N, 8]取出某一道题
    question_labels = (
        labels_matrix[
            :,
            question_index,
        ]
    )

    # 统计0、1、2、3各有多少样本
    class_counts = torch.bincount(
        question_labels,
        minlength=NUM_CLASSES,
    ).float()

    if torch.any(
        class_counts == 0
    ):
        raise ValueError(
            "训练集中存在样本数为0的类别，"
            "无法计算类别权重"
        )

    total_count = float(
        question_labels.numel()
    )

    # 原始权重：
    # 总样本数 / 当前类别样本数
    inverse_frequency = (
        total_count
        / class_counts
    )

    # 论文中的w(y)^beta
    class_weights = (
        inverse_frequency.pow(
            beta
        )
    )

    return (
        class_counts,
        class_weights,
    )

In [81]:
question_class_counts = {}
question_class_weights = {}

weight_table_rows = []


for question_index, question_name in enumerate(
    PHQ8_QUESTION_COLUMNS
):
    (
        class_counts,
        class_weights,
    ) = compute_question_class_weights(
        labels_matrix=(
            train_dataset.labels
        ),
        question_index=(
            question_index
        ),
        beta=BETA,
    )

    question_class_counts[
        question_name
    ] = class_counts

    question_class_weights[
        question_name
    ] = class_weights

    weight_table_rows.append(
        {
            "Question_Number": (
                question_index + 1
            ),
            "Question": question_name,

            "Count_0": (
                class_counts[0].item()
            ),
            "Count_1": (
                class_counts[1].item()
            ),
            "Count_2": (
                class_counts[2].item()
            ),
            "Count_3": (
                class_counts[3].item()
            ),

            "Weight_0": (
                class_weights[0].item()
            ),
            "Weight_1": (
                class_weights[1].item()
            ),
            "Weight_2": (
                class_weights[2].item()
            ),
            "Weight_3": (
                class_weights[3].item()
            ),
        }
    )


question_weight_df = pd.DataFrame(
    weight_table_rows
)


display(
    question_weight_df.round(4)
)

,Question_Number,Question,Count_0,Count_1,Count_2,Count_3,Weight_0,Weight_1,Weight_2,Weight_3
0,1,PHQ_8NoInterest,81.0,57.0,20.0,5.0,1.4186,1.6910,2.8548,5.7096
1,2,PHQ_8Depressed,68.0,63.0,18.0,14.0,1.5482,1.6085,3.0092,3.4122
2,3,PHQ_8Sleep,69.0,40.0,27.0,27.0,1.5370,2.0187,2.4570,2.4570
3,4,PHQ_8Tired,50.0,65.0,27.0,21.0,1.8055,1.5836,2.4570,2.7860
4,5,PHQ_8Appetite,77.0,42.0,24.0,20.0,1.4550,1.9700,2.6061,2.8548
5,6,PHQ_8Failure,76.0,44.0,24.0,19.0,1.4645,1.9247,2.6061,2.9290
6,7,PHQ_8Concentrating,93.0,37.0,14.0,19.0,1.3239,2.0989,3.4122,2.9290
7,8,PHQ_8Moving,128.0,20.0,9.0,6.0,1.1285,2.8548,4.2557,5.2122


In [82]:
def imbalanced_ordinal_loss(logits, labels, class_weights, alpha):
    if logits.ndim != 2:
        raise ValueError("logits必须是二维张量")

    if labels.ndim != 1:
        raise ValueError("labels必须是一维张量")

    if logits.shape[0] != labels.shape[0]:
        raise ValueError("logits与labels样本数不一致")

    if logits.shape[1] != NUM_CLASSES:
        raise ValueError("logits类别数量错误")

    labels = labels.long()

    class_weights = class_weights.to(device=logits.device, dtype=logits.dtype)

    # [batch, 4]
    probabilities = F.softmax(logits, dim=1)

    # 避免概率恰好为1，从而出现log(0)
    probabilities = probabilities.clamp(min=EPS, max=1.0 - EPS)

    # [1, 4]
    class_ids = torch.arange(NUM_CLASSES, device=logits.device).unsqueeze(0)

    # labels.unsqueeze(1)：[batch] → [batch, 1]
    # 广播相减后：[batch, 4]
    ordinal_distances = torch.abs(labels.unsqueeze(1) - class_ids).to(dtype=logits.dtype)

    # d(y,i)^alpha
    powered_distances = ordinal_distances.pow(alpha)

    # 取得每位样本真实类别的权重：[batch] → [batch, 1]
    sample_weights = class_weights[labels].unsqueeze(1)

    # -log(1-p_i)
    probability_penalties = -torch.log(1.0 - probabilities)

    # 每个错误类别分别产生损失，形状为[batch, 4]
    per_class_loss = probability_penalties * powered_distances * sample_weights

    # 先合并每个样本的4类损失
    per_sample_loss = per_class_loss.sum(dim=1)

    # 再对当前Batch取平均
    loss = per_sample_loss.mean()

    return loss


In [83]:
def compute_metrics(predictions, labels):
    predictions = predictions.detach().cpu().long().reshape(-1)
    labels = labels.detach().cpu().long().reshape(-1)

    if predictions.shape != labels.shape:
        raise ValueError("预测与标签形状不一致")

    if len(labels) == 0:
        raise ValueError("不能对空数据计算指标")

    # 行表示真实类别，列表示预测类别
    confusion_matrix = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

    for true_label, predicted_label in zip(labels, predictions):
        confusion_matrix[true_label, predicted_label] += 1

    confusion_float = confusion_matrix.float()

    # 对角线是各类别预测正确的数量
    true_positive = torch.diag(confusion_float)

    # 每一列之和：预测成该类别的数量
    predicted_count = confusion_float.sum(dim=0)

    # 每一行之和：该真实类别的数量
    support = confusion_float.sum(dim=1)

    precision = true_positive / predicted_count.clamp(min=EPS)
    recall = true_positive / support.clamp(min=EPS)
    f1_per_class = 2.0 * precision * recall / (precision + recall).clamp(min=EPS)
    macro_f1 = f1_per_class.mean()
    weighted_f1 = (f1_per_class * support).sum() / support.sum().clamp(min=EPS)

    total_true_positive = true_positive.sum()
    total_samples = support.sum()

    micro_precision = total_true_positive / total_samples.clamp(min=EPS)
    micro_recall = total_true_positive / total_samples.clamp(min=EPS)
    micro_f1 = 2.0 * micro_precision * micro_recall / (micro_precision + micro_recall).clamp(min=EPS)
    accuracy = total_true_positive / total_samples.clamp(min=EPS)

    # ---------- 序数指标 ----------
    predictions_float = predictions.float()
    labels_float = labels.float()

    prediction_mean = predictions_float.mean()
    label_mean = labels_float.mean()

    # correction=1表示样本方差
    prediction_variance = predictions_float.var(correction=1)
    label_variance = labels_float.var(correction=1)

    centered_predictions = predictions_float - prediction_mean
    centered_labels = labels_float - label_mean

    covariance = (centered_predictions * centered_labels).sum() / max(len(labels) - 1, 1)

    ccc = 2.0 * covariance / (prediction_variance + label_variance + (prediction_mean - label_mean).pow(2) + EPS)

    rmse = torch.sqrt(F.mse_loss(predictions_float, labels_float))
    mae = F.l1_loss(predictions_float, labels_float)

    return {
        "accuracy": accuracy.item(),
        "micro_f1": micro_f1.item(),
        "macro_f1": macro_f1.item(),
        "weighted_f1": weighted_f1.item(),
        "ccc": ccc.item(),
        "rmse": rmse.item(),
        "mae": mae.item(),
        "precision_per_class": precision,
        "recall_per_class": recall,
        "f1_per_class": f1_per_class,
        "support": support,
        "confusion_matrix": confusion_matrix,
    }

In [84]:
set_seed(42)


loss_check_model = (
    LSTMAttentionClassifier()
    .to(device)
)


loss_check_model.train()


real_embeddings = (
    batch_embeddings.to(device)
)

real_masks = (
    batch_masks.to(device)
)

real_labels = (
    batch_all_labels[
        :,
        0,
    ]
    .to(device)
    .long()
)


q1_class_weights = (
    question_class_weights[
        PHQ8_QUESTION_COLUMNS[0]
    ]
    .to(device)
)


loss_check_model.zero_grad(
    set_to_none=True
)


real_logits = loss_check_model(
    real_embeddings,
    real_masks,
)


real_loss = (
    imbalanced_ordinal_loss(
        logits=real_logits,
        labels=real_labels,
        class_weights=(
            q1_class_weights
        ),
        alpha=ALPHA,
    )
)


real_loss.backward()


gradient_norm_before_clipping = (
    torch.nn.utils.clip_grad_norm_(
        loss_check_model.parameters(),
        max_norm=MAX_GRAD_NORM,
    )
)


real_predictions = torch.argmax(
    real_logits,
    dim=1,
)


real_batch_metrics = (
    compute_metrics(
        predictions=(
            real_predictions
        ),
        labels=real_labels,
    )
)

设置随机种子：42


In [85]:
print(
    "Logits形状：",
    real_logits.shape,
)

print(
    "标签形状：",
    real_labels.shape,
)

print(
    "Q1类别权重：",
    q1_class_weights,
)

print(
    "ImbOLL形状：",
    real_loss.shape,
)

print(
    "ImbOLL数值：",
    real_loss.item(),
)

print(
    "损失是否为有限数值：",
    torch.isfinite(
        real_loss
    ).item(),
)

print(
    "裁剪前梯度长度：",
    gradient_norm_before_clipping.item(),
)

print(
    "当前Batch预测：",
    real_predictions,
)

print(
    "当前Batch真实标签：",
    real_labels,
)

print(
    "当前Batch Accuracy：",
    real_batch_metrics[
        "accuracy"
    ],
)

print(
    "当前Batch CCC：",
    real_batch_metrics[
        "ccc"
    ],
)

print(
    "当前Batch混淆矩阵：\n",
    real_batch_metrics[
        "confusion_matrix"
    ],
)

Logits形状： torch.Size([10, 4])
标签形状： torch.Size([10])
Q1类别权重： tensor([1.4186, 1.6910, 2.8548, 5.7096], device='cuda:0')
ImbOLL形状： torch.Size([])
ImbOLL数值： 3.2506158351898193
损失是否为有限数值： True
裁剪前梯度长度： 0.8588483333587646
当前Batch预测： tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')
当前Batch真实标签： tensor([1, 0, 0, 0, 0, 2, 2, 1, 3, 0], device='cuda:0')
当前Batch Accuracy： 0.20000000298023224
当前Batch CCC： 0.0
当前Batch混淆矩阵：
 tensor([[0, 5, 0, 0],
        [0, 2, 0, 0],
        [0, 2, 0, 0],
        [0, 1, 0, 0]])


In [86]:
del loss_check_model
del real_embeddings
del real_masks
del real_labels
del real_logits
del real_loss
del real_predictions


if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("损失检查模型已释放")

损失检查模型已释放


In [87]:
def create_dataloaders(
    seed,
):
    # 单独控制训练集打乱顺序
    train_generator = (
        torch.Generator()
    )

    train_generator.manual_seed(
        seed
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        drop_last=False,
        generator=train_generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )

    return (
        train_loader,
        val_loader,
    )

In [88]:
def train_one_epoch(
    model,
    dataloader,
    optimizer,
    question_index,
    class_weights,
    alpha,
    device,
    max_grad_norm,
    show_progress=False,
):
    model.train()

    total_loss = 0.0
    total_gradient_norm = 0.0

    all_predictions = []
    all_labels = []

    start_time = time.time()

    for batch_index, batch in enumerate(
        dataloader,
        start=1,
    ):
        (
            embeddings,
            key_padding_mask,
            all_question_labels,
        ) = batch

        # 从[B,8]中取出当前题目
        labels = all_question_labels[
            :,
            question_index,
        ]

        embeddings = (
            embeddings.to(device)
        )

        key_padding_mask = (
            key_padding_mask.to(
                device
            )
        )

        labels = (
            labels
            .to(device)
            .long()
        )

        # 清除上一个Batch的梯度
        optimizer.zero_grad(
            set_to_none=True
        )

        # 前向传播
        logits = model(
            embeddings,
            key_padding_mask,
        )

        # 当前题目的ImbOLL
        loss = (
            imbalanced_ordinal_loss(
                logits=logits,
                labels=labels,
                class_weights=(
                    class_weights
                ),
                alpha=alpha,
            )
        )

        # 反向传播
        loss.backward()

        # 返回值是裁剪之前的总梯度长度
        gradient_norm = (
            torch.nn.utils
            .clip_grad_norm_(
                model.parameters(),
                max_norm=(
                    max_grad_norm
                ),
            )
        )

        # AdamW更新参数
        optimizer.step()

        batch_size = (
            labels.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_gradient_norm += (
            gradient_norm.item()
        )

        predictions = torch.argmax(
            logits,
            dim=1,
        )

        all_predictions.append(
            predictions
            .detach()
            .cpu()
        )

        all_labels.append(
            labels
            .detach()
            .cpu()
        )

        if show_progress:
            print(
                f"\r训练Batch "
                f"{batch_index}/"
                f"{len(dataloader)} "
                f"Loss="
                f"{loss.item():.6f}",
                end="",
            )

    if show_progress:
        print()

    predictions = torch.cat(
        all_predictions,
        dim=0,
    )

    labels = torch.cat(
        all_labels,
        dim=0,
    )

    metrics = compute_metrics(
        predictions=predictions,
        labels=labels,
    )

    metrics["loss"] = (
        total_loss
        / len(dataloader.dataset)
    )

    metrics[
        "average_gradient_norm"
    ] = (
        total_gradient_norm
        / len(dataloader)
    )

    metrics["elapsed_seconds"] = (
        time.time()
        - start_time
    )

    return metrics

In [89]:
def evaluate_question(
    model,
    dataloader,
    question_index,
    class_weights,
    alpha,
    device,
    show_progress=False,
):
    model.eval()

    total_loss = 0.0

    all_predictions = []
    all_labels = []

    start_time = time.time()

    with torch.no_grad():

        for batch_index, batch in enumerate(
            dataloader,
            start=1,
        ):
            (
                embeddings,
                key_padding_mask,
                all_question_labels,
            ) = batch

            labels = (
                all_question_labels[
                    :,
                    question_index,
                ]
            )

            embeddings = (
                embeddings.to(device)
            )

            key_padding_mask = (
                key_padding_mask.to(
                    device
                )
            )

            labels = (
                labels
                .to(device)
                .long()
            )

            logits = model(
                embeddings,
                key_padding_mask,
            )

            loss = (
                imbalanced_ordinal_loss(
                    logits=logits,
                    labels=labels,
                    class_weights=(
                        class_weights
                    ),
                    alpha=alpha,
                )
            )

            batch_size = (
                labels.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            predictions = torch.argmax(
                logits,
                dim=1,
            )

            all_predictions.append(
                predictions.cpu()
            )

            all_labels.append(
                labels.cpu()
            )

            if show_progress:
                print(
                    f"\r验证Batch "
                    f"{batch_index}/"
                    f"{len(dataloader)}",
                    end="",
                )

    if show_progress:
        print()

    predictions = torch.cat(
        all_predictions,
        dim=0,
    )

    labels = torch.cat(
        all_labels,
        dim=0,
    )

    metrics = compute_metrics(
        predictions=predictions,
        labels=labels,
    )

    metrics["loss"] = (
        total_loss
        / len(dataloader.dataset)
    )

    metrics["elapsed_seconds"] = (
        time.time()
        - start_time
    )

    return metrics

In [90]:
def print_epoch_line(
    epoch,
    num_epochs,
    train_metrics,
    val_metrics,
):
    print(
        f"Epoch "
        f"{epoch:02d}/"
        f"{num_epochs:02d} | "
        f"Train Loss="
        f"{train_metrics['loss']:.4f} | "
        f"Val Loss="
        f"{val_metrics['loss']:.4f} | "
        f"Val Acc="
        f"{val_metrics['accuracy']:.4f} | "
        f"Val MacroF1="
        f"{val_metrics['macro_f1']:.4f} | "
        f"Val CCC="
        f"{val_metrics['ccc']:.4f}"
    )

In [91]:
def metrics_to_cpu(
    metrics,
):
    copied_metrics = {}

    for key, value in (
        metrics.items()
    ):
        if torch.is_tensor(value):
            copied_metrics[key] = (
                value
                .detach()
                .cpu()
            )
        else:
            copied_metrics[key] = (
                value
            )

    return copied_metrics

In [92]:
def save_training_checkpoint(
    checkpoint_path,
    checkpoint_type,
    epoch,
    model,
    optimizer,
    train_metrics,
    val_metrics,
    question_index,
    question_name,
    seed,
    class_counts,
    class_weights,
    best_val_loss,
    best_val_ccc,
    best_loss_epoch,
    best_ccc_epoch,
    include_optimizer,
):
    checkpoint = {
        "checkpoint_type": (
            checkpoint_type
        ),
        "epoch": epoch,

        "question_index": (
            question_index
        ),
        "question_number": (
            question_index + 1
        ),
        "question_name": (
            question_name
        ),
        "seed": seed,

        "model_state_dict": (
            model.state_dict()
        ),

        "train_metrics": (
            metrics_to_cpu(
                train_metrics
            )
        ),

        "val_metrics": (
            metrics_to_cpu(
                val_metrics
            )
        ),

        "class_counts": (
            class_counts
            .detach()
            .cpu()
        ),

        "class_weights": (
            class_weights
            .detach()
            .cpu()
        ),

        "best_val_loss": (
            best_val_loss
        ),
        "best_val_ccc": (
            best_val_ccc
        ),
        "best_loss_epoch": (
            best_loss_epoch
        ),
        "best_ccc_epoch": (
            best_ccc_epoch
        ),

        "model_name": MODEL_NAME,
        "model_revision": (
            MODEL_REVISION
        ),

        "max_turns": MAX_TURNS,
        "embedding_dim": (
            EMBEDDING_DIM
        ),
        "lstm_hidden_size": (
            LSTM_HIDDEN_SIZE
        ),
        "num_attention_heads": (
            NUM_ATTENTION_HEADS
        ),
        "num_classes": (
            NUM_CLASSES
        ),

        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": (
            LEARNING_RATE
        ),
        "optimizer_eps": (
            OPTIMIZER_EPS
        ),
        "weight_decay": (
            WEIGHT_DECAY
        ),
        "max_grad_norm": (
            MAX_GRAD_NORM
        ),
        "alpha": ALPHA,
        "beta": BETA,
    }

    if include_optimizer:
        checkpoint[
            "optimizer_state_dict"
        ] = optimizer.state_dict()

    torch.save(
        checkpoint,
        checkpoint_path,
    )

In [93]:
def train_single_question(
    question_index,
    seed,
    num_epochs=NUM_EPOCHS,
    show_batch_progress=False,
):
    if question_index < 0 or (
        question_index
        >= len(
            PHQ8_QUESTION_COLUMNS
        )
    ):
        raise ValueError(
            f"非法question_index："
            f"{question_index}"
        )

    question_number = (
        question_index + 1
    )

    question_name = (
        PHQ8_QUESTION_COLUMNS[
            question_index
        ]
    )

    print()
    print("=" * 70)
    print(
        f"开始训练 Q{question_number}："
        f"{question_name}"
    )
    print(
        f"随机种子：{seed}"
    )
    print("=" * 70)

    # 每个实验重新设置随机状态
    set_seed(seed)

    # 每个实验重新创建DataLoader
    (
        train_loader,
        val_loader,
    ) = create_dataloaders(
        seed=seed
    )

    # 每个实验重新创建独立模型
    model = (
        LSTMAttentionClassifier()
        .to(device)
    )

    # 严格使用公开代码中的AdamW
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        eps=OPTIMIZER_EPS,
        weight_decay=WEIGHT_DECAY,
    )

    # 当前题目的类别数量和权重
    (
        class_counts,
        class_weights,
    ) = compute_question_class_weights(
        labels_matrix=(
            train_dataset.labels
        ),
        question_index=(
            question_index
        ),
        beta=BETA,
    )

    class_weights = (
        class_weights.to(device)
    )

    # 当前实验单独建立目录
    experiment_dir = (
        CHECKPOINT_ROOT
        / (
            f"q{question_number}_"
            f"{question_name}"
        )
        / f"seed_{seed}"
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    best_loss_path = (
        experiment_dir
        / "best_loss.pt"
    )

    best_ccc_path = (
        experiment_dir
        / "best_ccc.pt"
    )

    last_path = (
        experiment_dir
        / "last.pt"
    )

    history_path = (
        experiment_dir
        / "history.csv"
    )

    best_val_loss = float(
        "inf"
    )

    best_val_ccc = -float(
        "inf"
    )

    best_loss_epoch = None
    best_ccc_epoch = None

    history_rows = []

    experiment_start_time = (
        time.time()
    )

    for epoch in range(
        1,
        num_epochs + 1,
    ):
        train_metrics = (
            train_one_epoch(
                model=model,
                dataloader=(
                    train_loader
                ),
                optimizer=optimizer,
                question_index=(
                    question_index
                ),
                class_weights=(
                    class_weights
                ),
                alpha=ALPHA,
                device=device,
                max_grad_norm=(
                    MAX_GRAD_NORM
                ),
                show_progress=(
                    show_batch_progress
                ),
            )
        )

        val_metrics = (
            evaluate_question(
                model=model,
                dataloader=(
                    val_loader
                ),
                question_index=(
                    question_index
                ),
                class_weights=(
                    class_weights
                ),
                alpha=ALPHA,
                device=device,
                show_progress=False,
            )
        )

        # 判断本轮是否产生新最佳结果
        improved_loss = (
            val_metrics["loss"]
            < best_val_loss
        )

        improved_ccc = (
            val_metrics["ccc"]
            > best_val_ccc
        )

        if improved_loss:
            best_val_loss = (
                val_metrics["loss"]
            )

            best_loss_epoch = (
                epoch
            )

        if improved_ccc:
            best_val_ccc = (
                val_metrics["ccc"]
            )

            best_ccc_epoch = (
                epoch
            )

        history_row = {
            "epoch": epoch,

            "train_loss": (
                train_metrics["loss"]
            ),
            "val_loss": (
                val_metrics["loss"]
            ),

            "train_accuracy": (
                train_metrics[
                    "accuracy"
                ]
            ),
            "val_accuracy": (
                val_metrics[
                    "accuracy"
                ]
            ),

            "train_macro_f1": (
                train_metrics[
                    "macro_f1"
                ]
            ),
            "val_macro_f1": (
                val_metrics[
                    "macro_f1"
                ]
            ),

            "train_ccc": (
                train_metrics["ccc"]
            ),
            "val_ccc": (
                val_metrics["ccc"]
            ),

            "val_rmse": (
                val_metrics["rmse"]
            ),
            "val_mae": (
                val_metrics["mae"]
            ),

            "average_gradient_norm": (
                train_metrics[
                    "average_gradient_norm"
                ]
            ),

            "train_seconds": (
                train_metrics[
                    "elapsed_seconds"
                ]
            ),
            "val_seconds": (
                val_metrics[
                    "elapsed_seconds"
                ]
            ),
        }

        history_rows.append(
            history_row
        )

        # 每轮覆盖保存历史CSV
        pd.DataFrame(
            history_rows
        ).to_csv(
            history_path,
            index=False,
        )

        # 保存新的最佳Loss模型
        if improved_loss:
            save_training_checkpoint(
                checkpoint_path=(
                    best_loss_path
                ),
                checkpoint_type=(
                    "best_loss"
                ),
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                train_metrics=(
                    train_metrics
                ),
                val_metrics=(
                    val_metrics
                ),
                question_index=(
                    question_index
                ),
                question_name=(
                    question_name
                ),
                seed=seed,
                class_counts=(
                    class_counts
                ),
                class_weights=(
                    class_weights
                ),
                best_val_loss=(
                    best_val_loss
                ),
                best_val_ccc=(
                    best_val_ccc
                ),
                best_loss_epoch=(
                    best_loss_epoch
                ),
                best_ccc_epoch=(
                    best_ccc_epoch
                ),
                include_optimizer=False,
            )

        # 保存新的最佳CCC模型
        if improved_ccc:
            save_training_checkpoint(
                checkpoint_path=(
                    best_ccc_path
                ),
                checkpoint_type=(
                    "best_ccc"
                ),
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                train_metrics=(
                    train_metrics
                ),
                val_metrics=(
                    val_metrics
                ),
                question_index=(
                    question_index
                ),
                question_name=(
                    question_name
                ),
                seed=seed,
                class_counts=(
                    class_counts
                ),
                class_weights=(
                    class_weights
                ),
                best_val_loss=(
                    best_val_loss
                ),
                best_val_ccc=(
                    best_val_ccc
                ),
                best_loss_epoch=(
                    best_loss_epoch
                ),
                best_ccc_epoch=(
                    best_ccc_epoch
                ),
                include_optimizer=False,
            )

        # last每轮都覆盖，用于恢复
        save_training_checkpoint(
            checkpoint_path=(
                last_path
            ),
            checkpoint_type="last",
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            train_metrics=(
                train_metrics
            ),
            val_metrics=(
                val_metrics
            ),
            question_index=(
                question_index
            ),
            question_name=(
                question_name
            ),
            seed=seed,
            class_counts=(
                class_counts
            ),
            class_weights=(
                class_weights
            ),
            best_val_loss=(
                best_val_loss
            ),
            best_val_ccc=(
                best_val_ccc
            ),
            best_loss_epoch=(
                best_loss_epoch
            ),
            best_ccc_epoch=(
                best_ccc_epoch
            ),
            include_optimizer=True,
        )

        print_epoch_line(
            epoch=epoch,
            num_epochs=num_epochs,
            train_metrics=(
                train_metrics
            ),
            val_metrics=(
                val_metrics
            ),
        )

        saved_messages = []

        if improved_loss:
            saved_messages.append(
                "best_loss"
            )

        if improved_ccc:
            saved_messages.append(
                "best_ccc"
            )

        if saved_messages:
            print(
                "  保存：",
                ", ".join(
                    saved_messages
                ),
            )

    total_seconds = (
        time.time()
        - experiment_start_time
    )

    history_df = pd.DataFrame(
        history_rows
    )

    result = {
        "question_index": (
            question_index
        ),
        "question_number": (
            question_number
        ),
        "question_name": (
            question_name
        ),
        "seed": seed,

        "best_val_loss": (
            best_val_loss
        ),
        "best_loss_epoch": (
            best_loss_epoch
        ),

        "best_val_ccc": (
            best_val_ccc
        ),
        "best_ccc_epoch": (
            best_ccc_epoch
        ),

        "best_loss_path": (
            best_loss_path
        ),
        "best_ccc_path": (
            best_ccc_path
        ),
        "last_path": last_path,
        "history_path": (
            history_path
        ),

        "history": history_df,
        "elapsed_seconds": (
            total_seconds
        ),
    }

    print()
    print(
        f"Q{question_number} "
        f"seed={seed}训练结束"
    )

    print(
        f"最佳Loss："
        f"{best_val_loss:.6f}，"
        f"Epoch {best_loss_epoch}"
    )

    print(
        f"最佳CCC："
        f"{best_val_ccc:.6f}，"
        f"Epoch {best_ccc_epoch}"
    )

    print(
        f"总耗时："
        f"{total_seconds:.2f}秒"
    )

    # 返回前先释放GPU模型
    del model
    del optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [94]:
def train_single_question(
    question_index,
    seed,
    num_epochs=NUM_EPOCHS,
    show_batch_progress=False,
):
    if question_index < 0 or (
        question_index
        >= len(
            PHQ8_QUESTION_COLUMNS
        )
    ):
        raise ValueError(
            f"非法question_index："
            f"{question_index}"
        )

    question_number = (
        question_index + 1
    )

    question_name = (
        PHQ8_QUESTION_COLUMNS[
            question_index
        ]
    )

    print()
    print("=" * 70)
    print(
        f"开始训练 Q{question_number}："
        f"{question_name}"
    )
    print(
        f"随机种子：{seed}"
    )
    print("=" * 70)

    # 每个实验重新设置随机状态
    set_seed(seed)

    # 每个实验重新创建DataLoader
    (
        train_loader,
        val_loader,
    ) = create_dataloaders(
        seed=seed
    )

    # 每个实验重新创建独立模型
    model = (
        LSTMAttentionClassifier()
        .to(device)
    )

    # 严格使用公开代码中的AdamW
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        eps=OPTIMIZER_EPS,
        weight_decay=WEIGHT_DECAY,
    )

    # 当前题目的类别数量和权重
    (
        class_counts,
        class_weights,
    ) = compute_question_class_weights(
        labels_matrix=(
            train_dataset.labels
        ),
        question_index=(
            question_index
        ),
        beta=BETA,
    )

    class_weights = (
        class_weights.to(device)
    )

    # 当前实验单独建立目录
    experiment_dir = (
        CHECKPOINT_ROOT
        / (
            f"q{question_number}_"
            f"{question_name}"
        )
        / f"seed_{seed}"
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    best_loss_path = (
        experiment_dir
        / "best_loss.pt"
    )

    best_ccc_path = (
        experiment_dir
        / "best_ccc.pt"
    )

    last_path = (
        experiment_dir
        / "last.pt"
    )

    history_path = (
        experiment_dir
        / "history.csv"
    )

    best_val_loss = float(
        "inf"
    )

    best_val_ccc = -float(
        "inf"
    )

    best_loss_epoch = None
    best_ccc_epoch = None

    history_rows = []

    experiment_start_time = (
        time.time()
    )

    for epoch in range(
        1,
        num_epochs + 1,
    ):
        train_metrics = (
            train_one_epoch(
                model=model,
                dataloader=(
                    train_loader
                ),
                optimizer=optimizer,
                question_index=(
                    question_index
                ),
                class_weights=(
                    class_weights
                ),
                alpha=ALPHA,
                device=device,
                max_grad_norm=(
                    MAX_GRAD_NORM
                ),
                show_progress=(
                    show_batch_progress
                ),
            )
        )

        val_metrics = (
            evaluate_question(
                model=model,
                dataloader=(
                    val_loader
                ),
                question_index=(
                    question_index
                ),
                class_weights=(
                    class_weights
                ),
                alpha=ALPHA,
                device=device,
                show_progress=False,
            )
        )

        # 判断本轮是否产生新最佳结果
        improved_loss = (
            val_metrics["loss"]
            < best_val_loss
        )

        improved_ccc = (
            val_metrics["ccc"]
            > best_val_ccc
        )

        if improved_loss:
            best_val_loss = (
                val_metrics["loss"]
            )

            best_loss_epoch = (
                epoch
            )

        if improved_ccc:
            best_val_ccc = (
                val_metrics["ccc"]
            )

            best_ccc_epoch = (
                epoch
            )

        history_row = {
            "epoch": epoch,

            "train_loss": (
                train_metrics["loss"]
            ),
            "val_loss": (
                val_metrics["loss"]
            ),

            "train_accuracy": (
                train_metrics[
                    "accuracy"
                ]
            ),
            "val_accuracy": (
                val_metrics[
                    "accuracy"
                ]
            ),

            "train_macro_f1": (
                train_metrics[
                    "macro_f1"
                ]
            ),
            "val_macro_f1": (
                val_metrics[
                    "macro_f1"
                ]
            ),

            "train_ccc": (
                train_metrics["ccc"]
            ),
            "val_ccc": (
                val_metrics["ccc"]
            ),

            "val_rmse": (
                val_metrics["rmse"]
            ),
            "val_mae": (
                val_metrics["mae"]
            ),

            "average_gradient_norm": (
                train_metrics[
                    "average_gradient_norm"
                ]
            ),

            "train_seconds": (
                train_metrics[
                    "elapsed_seconds"
                ]
            ),
            "val_seconds": (
                val_metrics[
                    "elapsed_seconds"
                ]
            ),
        }

        history_rows.append(
            history_row
        )

        # 每轮覆盖保存历史CSV
        pd.DataFrame(
            history_rows
        ).to_csv(
            history_path,
            index=False,
        )

        # 保存新的最佳Loss模型
        if improved_loss:
            save_training_checkpoint(
                checkpoint_path=(
                    best_loss_path
                ),
                checkpoint_type=(
                    "best_loss"
                ),
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                train_metrics=(
                    train_metrics
                ),
                val_metrics=(
                    val_metrics
                ),
                question_index=(
                    question_index
                ),
                question_name=(
                    question_name
                ),
                seed=seed,
                class_counts=(
                    class_counts
                ),
                class_weights=(
                    class_weights
                ),
                best_val_loss=(
                    best_val_loss
                ),
                best_val_ccc=(
                    best_val_ccc
                ),
                best_loss_epoch=(
                    best_loss_epoch
                ),
                best_ccc_epoch=(
                    best_ccc_epoch
                ),
                include_optimizer=False,
            )

        # 保存新的最佳CCC模型
        if improved_ccc:
            save_training_checkpoint(
                checkpoint_path=(
                    best_ccc_path
                ),
                checkpoint_type=(
                    "best_ccc"
                ),
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                train_metrics=(
                    train_metrics
                ),
                val_metrics=(
                    val_metrics
                ),
                question_index=(
                    question_index
                ),
                question_name=(
                    question_name
                ),
                seed=seed,
                class_counts=(
                    class_counts
                ),
                class_weights=(
                    class_weights
                ),
                best_val_loss=(
                    best_val_loss
                ),
                best_val_ccc=(
                    best_val_ccc
                ),
                best_loss_epoch=(
                    best_loss_epoch
                ),
                best_ccc_epoch=(
                    best_ccc_epoch
                ),
                include_optimizer=False,
            )

        # last每轮都覆盖，用于恢复
        save_training_checkpoint(
            checkpoint_path=(
                last_path
            ),
            checkpoint_type="last",
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            train_metrics=(
                train_metrics
            ),
            val_metrics=(
                val_metrics
            ),
            question_index=(
                question_index
            ),
            question_name=(
                question_name
            ),
            seed=seed,
            class_counts=(
                class_counts
            ),
            class_weights=(
                class_weights
            ),
            best_val_loss=(
                best_val_loss
            ),
            best_val_ccc=(
                best_val_ccc
            ),
            best_loss_epoch=(
                best_loss_epoch
            ),
            best_ccc_epoch=(
                best_ccc_epoch
            ),
            include_optimizer=True,
        )

        print_epoch_line(
            epoch=epoch,
            num_epochs=num_epochs,
            train_metrics=(
                train_metrics
            ),
            val_metrics=(
                val_metrics
            ),
        )

        saved_messages = []

        if improved_loss:
            saved_messages.append(
                "best_loss"
            )

        if improved_ccc:
            saved_messages.append(
                "best_ccc"
            )

        if saved_messages:
            print(
                "  保存：",
                ", ".join(
                    saved_messages
                ),
            )

    total_seconds = (
        time.time()
        - experiment_start_time
    )

    history_df = pd.DataFrame(
        history_rows
    )

    result = {
        "question_index": (
            question_index
        ),
        "question_number": (
            question_number
        ),
        "question_name": (
            question_name
        ),
        "seed": seed,

        "best_val_loss": (
            best_val_loss
        ),
        "best_loss_epoch": (
            best_loss_epoch
        ),

        "best_val_ccc": (
            best_val_ccc
        ),
        "best_ccc_epoch": (
            best_ccc_epoch
        ),

        "best_loss_path": (
            best_loss_path
        ),
        "best_ccc_path": (
            best_ccc_path
        ),
        "last_path": last_path,
        "history_path": (
            history_path
        ),

        "history": history_df,
        "elapsed_seconds": (
            total_seconds
        ),
    }

    print()
    print(
        f"Q{question_number} "
        f"seed={seed}训练结束"
    )

    print(
        f"最佳Loss："
        f"{best_val_loss:.6f}，"
        f"Epoch {best_loss_epoch}"
    )

    print(
        f"最佳CCC："
        f"{best_val_ccc:.6f}，"
        f"Epoch {best_ccc_epoch}"
    )

    print(
        f"总耗时："
        f"{total_seconds:.2f}秒"
    )

    # 返回前先释放GPU模型
    del model
    del optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [95]:
q1_seed42_result = (
    train_single_question(
        question_index=0,
        seed=42,
        num_epochs=NUM_EPOCHS,
        show_batch_progress=False,
    )
)


开始训练 Q1：PHQ_8NoInterest
随机种子：42
设置随机种子：42
Epoch 01/20 | Train Loss=2.3273 | Val Loss=2.6111 | Val Acc=0.4545 | Val MacroF1=0.1562 | Val CCC=0.0000
  保存： best_loss, best_ccc
Epoch 02/20 | Train Loss=2.1514 | Val Loss=2.5398 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
  保存： best_loss
Epoch 03/20 | Train Loss=2.1565 | Val Loss=2.5475 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 04/20 | Train Loss=2.1371 | Val Loss=2.5850 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 05/20 | Train Loss=2.0924 | Val Loss=2.4858 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
  保存： best_loss
Epoch 06/20 | Train Loss=2.0753 | Val Loss=2.4782 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
  保存： best_loss
Epoch 07/20 | Train Loss=1.9118 | Val Loss=2.3884 | Val Acc=0.4364 | Val MacroF1=0.3178 | Val CCC=0.2458
  保存： best_loss, best_ccc
Epoch 08/20 | Train Loss=1.6336 | Val Loss=2.4210 | Val Acc=0.6000 | Val MacroF1=0.4185 | Val CCC=0.4125
  保存： best_ccc
Ep

In [96]:
print(
    "Q1最佳CCC：",
    q1_seed42_result[
        "best_val_ccc"
    ],
)

print(
    "Q1最佳CCC轮数：",
    q1_seed42_result[
        "best_ccc_epoch"
    ],
)

print(
    "Q1最佳CCC文件：",
    q1_seed42_result[
        "best_ccc_path"
    ],
)

display(
    q1_seed42_result[
        "history"
    ].round(4)
)

Q1最佳CCC： 0.4533407390117645
Q1最佳CCC轮数： 9
Q1最佳CCC文件： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_ccc.pt


,epoch,train_loss,val_loss,train_accuracy,val_accuracy,train_macro_f1,val_macro_f1,train_ccc,val_ccc,val_rmse,val_mae,average_gradient_norm,train_seconds,val_seconds
0,1,2.3273,2.6111,0.3681,0.4545,0.1880,0.1562,-0.0679,0.0000,1.1599,0.7636,4.8375,0.8264,0.0742
1,2,2.1514,2.5398,0.3620,0.4000,0.1770,0.1429,0.0383,0.0000,0.9045,0.6727,4.3094,0.4307,0.0533
2,3,2.1565,2.5475,0.3436,0.4000,0.1504,0.1429,-0.0584,0.0000,0.9045,0.6727,3.6316,0.4333,0.0638
3,4,2.1371,2.5850,0.3681,0.4000,0.1653,0.1429,0.0148,0.0000,0.9045,0.6727,3.5886,0.3865,0.0537
4,5,2.0924,2.4858,0.3681,0.4000,0.1683,0.1429,0.0415,0.0000,0.9045,0.6727,4.5644,0.3433,0.0496
5,6,2.0753,2.4782,0.4172,0.4000,0.2260,0.1429,0.0593,0.0000,0.9045,0.6727,6.4846,0.3635,0.0561
6,7,1.9118,2.3884,0.4785,0.4364,0.2992,0.3178,0.2180,0.2458,0.9342,0.6545,5.9356,0.3463,0.0546
7,8,1.6336,2.4210,0.5828,0.6000,0.4301,0.4185,0.5606,0.4125,0.8421,0.4909,9.1156,0.3408,0.0536
8,9,1.6425,2.3725,0.6380,0.6000,0.4779,0.4187,0.6487,0.4533,0.8090,0.4727,18.2675,0.3380,0.0542
9,10,1.5594,2.5504,0.6074,0.3818,0.4403,0.2536,0.6178,0.2819,0.8842,0.6727,17.4946,0.3517,0.0852


In [97]:
def get_experiment_paths(
    question_index,
    seed,
):
    question_number = (
        question_index + 1
    )

    question_name = (
        PHQ8_QUESTION_COLUMNS[
            question_index
        ]
    )

    experiment_dir = (
        CHECKPOINT_ROOT
        / (
            f"q{question_number}_"
            f"{question_name}"
        )
        / f"seed_{seed}"
    )

    return {
        "experiment_dir": (
            experiment_dir
        ),
        "best_loss_path": (
            experiment_dir
            / "best_loss.pt"
        ),
        "best_ccc_path": (
            experiment_dir
            / "best_ccc.pt"
        ),
        "last_path": (
            experiment_dir
            / "last.pt"
        ),
        "history_path": (
            experiment_dir
            / "history.csv"
        ),
    }

In [98]:
def load_completed_experiment(
    question_index,
    seed,
    num_epochs=NUM_EPOCHS,
):
    paths = get_experiment_paths(
        question_index=question_index,
        seed=seed,
    )

    required_paths = [
        paths["best_loss_path"],
        paths["best_ccc_path"],
        paths["last_path"],
        paths["history_path"],
    ]

    if not all(
        path.exists()
        for path in required_paths
    ):
        return None

    last_checkpoint = torch.load(
        paths["last_path"],
        map_location="cpu",
        weights_only=False,
    )

    completed_epoch = int(
        last_checkpoint["epoch"]
    )

    if completed_epoch < num_epochs:
        return None

    history_df = pd.read_csv(
        paths["history_path"]
    )

    result = {
        "question_index": (
            question_index
        ),
        "question_number": (
            question_index + 1
        ),
        "question_name": (
            PHQ8_QUESTION_COLUMNS[
                question_index
            ]
        ),
        "seed": seed,

        "best_val_loss": (
            last_checkpoint[
                "best_val_loss"
            ]
        ),
        "best_loss_epoch": (
            last_checkpoint[
                "best_loss_epoch"
            ]
        ),

        "best_val_ccc": (
            last_checkpoint[
                "best_val_ccc"
            ]
        ),
        "best_ccc_epoch": (
            last_checkpoint[
                "best_ccc_epoch"
            ]
        ),

        "best_loss_path": (
            paths["best_loss_path"]
        ),
        "best_ccc_path": (
            paths["best_ccc_path"]
        ),
        "last_path": (
            paths["last_path"]
        ),
        "history_path": (
            paths["history_path"]
        ),

        "history": history_df,

        "elapsed_seconds": (
            history_df[
                "train_seconds"
            ].sum()
            + history_df[
                "val_seconds"
            ].sum()
        ),
    }

    return result

In [99]:
def train_or_load_question(
    question_index,
    seed,
):
    completed_result = (
        load_completed_experiment(
            question_index=(
                question_index
            ),
            seed=seed,
            num_epochs=NUM_EPOCHS,
        )
    )

    if completed_result is not None:
        print(
            f"Q{question_index + 1} "
            f"seed={seed}已经完成，"
            "直接读取已有结果"
        )

        return completed_result

    return train_single_question(
        question_index=(
            question_index
        ),
        seed=seed,
        num_epochs=NUM_EPOCHS,
        show_batch_progress=False,
    )

In [100]:
seed42_results = {
    0: q1_seed42_result
}


for question_index in range(
    1,
    len(
        PHQ8_QUESTION_COLUMNS
    ),
):
    seed42_results[
        question_index
    ] = train_or_load_question(
        question_index=(
            question_index
        ),
        seed=42,
    )


开始训练 Q2：PHQ_8Depressed
随机种子：42
设置随机种子：42
Epoch 01/20 | Train Loss=2.4726 | Val Loss=2.3490 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
  保存： best_loss, best_ccc
Epoch 02/20 | Train Loss=2.3910 | Val Loss=2.3236 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
  保存： best_loss
Epoch 03/20 | Train Loss=2.3768 | Val Loss=2.3317 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
Epoch 04/20 | Train Loss=2.3652 | Val Loss=2.3066 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
  保存： best_loss
Epoch 05/20 | Train Loss=2.3355 | Val Loss=2.2557 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
  保存： best_loss
Epoch 06/20 | Train Loss=2.2524 | Val Loss=2.2926 | Val Acc=0.3091 | Val MacroF1=0.2216 | Val CCC=0.3047
  保存： best_ccc
Epoch 07/20 | Train Loss=2.0319 | Val Loss=2.0747 | Val Acc=0.4727 | Val MacroF1=0.3616 | Val CCC=0.4414
  保存： best_loss, best_ccc
Epoch 08/20 | Train Loss=1.8698 | Val Loss=2.2206 | Val Acc=0.5455 | Val MacroF1=0.3304 | Val CCC=0.3663
Epo

In [101]:
seed42_summary_rows = []


for question_index in range(
    len(
        PHQ8_QUESTION_COLUMNS
    )
):
    result = seed42_results[
        question_index
    ]

    seed42_summary_rows.append(
        {
            "Question_Number": (
                result[
                    "question_number"
                ]
            ),
            "Question": (
                result[
                    "question_name"
                ]
            ),
            "Seed": (
                result["seed"]
            ),
            "Best_Loss": (
                result[
                    "best_val_loss"
                ]
            ),
            "Best_Loss_Epoch": (
                result[
                    "best_loss_epoch"
                ]
            ),
            "Best_CCC": (
                result[
                    "best_val_ccc"
                ]
            ),
            "Best_CCC_Epoch": (
                result[
                    "best_ccc_epoch"
                ]
            ),
            "Elapsed_Seconds": (
                result[
                    "elapsed_seconds"
                ]
            ),
        }
    )


seed42_summary_df = pd.DataFrame(
    seed42_summary_rows
)


seed42_summary_path = (
    CHECKPOINT_ROOT
    / "seed_42_question_summary.csv"
)


seed42_summary_df.to_csv(
    seed42_summary_path,
    index=False,
)


display(
    seed42_summary_df.round(4)
)


print(
    "Seed 42汇总已保存：",
    seed42_summary_path,
)

print(
    "8题平均单题CCC：",
    seed42_summary_df[
        "Best_CCC"
    ].mean(),
)

,Question_Number,Question,Seed,Best_Loss,Best_Loss_Epoch,Best_CCC,Best_CCC_Epoch,Elapsed_Seconds
0,1,PHQ_8NoInterest,42,2.2815,13,0.4533,9,10.1422
1,2,PHQ_8Depressed,42,1.9889,9,0.5286,11,10.1873
2,3,PHQ_8Sleep,42,2.2500,9,0.4160,15,9.4345
3,4,PHQ_8Tired,42,2.1974,11,0.4717,10,9.4370
4,5,PHQ_8Appetite,42,2.0702,13,0.5130,19,9.3743
5,6,PHQ_8Failure,42,2.0042,10,0.4985,14,9.6343
6,7,PHQ_8Concentrating,42,1.9630,15,0.5171,20,9.5935
7,8,PHQ_8Moving,42,1.6782,11,0.3374,12,9.2875


Seed 42汇总已保存： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_42_question_summary.csv
8题平均单题CCC： 0.46694478392601013


In [102]:
def load_question_model(
    checkpoint_path,
    expected_question_index,
    expected_seed,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    if checkpoint[
        "question_index"
    ] != expected_question_index:
        raise ValueError(
            "Checkpoint题目编号不一致"
        )

    if checkpoint[
        "seed"
    ] != expected_seed:
        raise ValueError(
            "Checkpoint随机种子不一致"
        )

    model = (
        LSTMAttentionClassifier()
        .to(device)
    )

    load_result = (
        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )
    )

    if load_result.missing_keys:
        raise ValueError(
            "缺少模型参数："
            f"{load_result.missing_keys}"
        )

    if load_result.unexpected_keys:
        raise ValueError(
            "存在多余模型参数："
            f"{load_result.unexpected_keys}"
        )

    model.eval()

    return (
        model,
        checkpoint,
    )

In [103]:
def predict_single_question(
    model,
    dataloader,
    question_index,
    device,
):
    all_probabilities = []
    all_predictions = []
    all_labels = []

    model.eval()

    with torch.no_grad():

        for batch in dataloader:
            (
                embeddings,
                key_padding_mask,
                all_question_labels,
            ) = batch

            labels = (
                all_question_labels[
                    :,
                    question_index,
                ]
                .long()
            )

            embeddings = (
                embeddings.to(device)
            )

            key_padding_mask = (
                key_padding_mask.to(
                    device
                )
            )

            logits = model(
                embeddings,
                key_padding_mask,
            )

            probabilities = F.softmax(
                logits,
                dim=1,
            )

            predictions = torch.argmax(
                probabilities,
                dim=1,
            )

            all_probabilities.append(
                probabilities.cpu()
            )

            all_predictions.append(
                predictions.cpu()
            )

            all_labels.append(
                labels.cpu()
            )

    probabilities = torch.cat(
        all_probabilities,
        dim=0,
    )

    predictions = torch.cat(
        all_predictions,
        dim=0,
    )

    labels = torch.cat(
        all_labels,
        dim=0,
    )

    return (
        probabilities,
        predictions,
        labels,
    )

In [104]:
total_score_val_loader = (
    DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )
)

In [105]:
validation_count = len(
    val_dataset
)


seed42_question_predictions = (
    torch.empty(
        (
            validation_count,
            len(
                PHQ8_QUESTION_COLUMNS
            ),
        ),
        dtype=torch.long,
    )
)


seed42_question_probabilities = (
    torch.empty(
        (
            validation_count,
            len(
                PHQ8_QUESTION_COLUMNS
            ),
            NUM_CLASSES,
        ),
        dtype=torch.float32,
    )
)


true_question_labels = (
    val_dataset.labels.clone()
)


restored_question_rows = []

In [106]:
for question_index in range(
    len(
        PHQ8_QUESTION_COLUMNS
    )
):
    result = seed42_results[
        question_index
    ]

    (
        question_model,
        question_checkpoint,
    ) = load_question_model(
        checkpoint_path=(
            result[
                "best_ccc_path"
            ]
        ),
        expected_question_index=(
            question_index
        ),
        expected_seed=42,
    )

    (
        probabilities,
        predictions,
        labels,
    ) = predict_single_question(
        model=question_model,
        dataloader=(
            total_score_val_loader
        ),
        question_index=(
            question_index
        ),
        device=device,
    )

    # 保存到对应题目列
    seed42_question_predictions[
        :,
        question_index,
    ] = predictions

    seed42_question_probabilities[
        :,
        question_index,
        :,
    ] = probabilities

    # 检查真实标签顺序是否一致
    if not torch.equal(
        labels,
        true_question_labels[
            :,
            question_index,
        ],
    ):
        raise RuntimeError(
            f"Q{question_index + 1}"
            "验证标签顺序不一致"
        )

    restored_metrics = (
        compute_metrics(
            predictions=predictions,
            labels=labels,
        )
    )

    saved_ccc = (
        question_checkpoint[
            "val_metrics"
        ]["ccc"]
    )

    ccc_difference = abs(
        restored_metrics["ccc"]
        - saved_ccc
    )

    restored_question_rows.append(
        {
            "Question_Number": (
                question_index + 1
            ),
            "Question": (
                PHQ8_QUESTION_COLUMNS[
                    question_index
                ]
            ),
            "Checkpoint_Epoch": (
                question_checkpoint[
                    "epoch"
                ]
            ),
            "Saved_CCC": (
                saved_ccc
            ),
            "Restored_CCC": (
                restored_metrics[
                    "ccc"
                ]
            ),
            "CCC_Difference": (
                ccc_difference
            ),
        }
    )

    print(
        f"Q{question_index + 1} "
        f"加载完成，"
        f"CCC="
        f"{restored_metrics['ccc']:.4f}"
    )

    del question_model
    del question_checkpoint

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Q1 加载完成，CCC=0.4533
Q2 加载完成，CCC=0.5286
Q3 加载完成，CCC=0.4160
Q4 加载完成，CCC=0.4717
Q5 加载完成，CCC=0.5130
Q6 加载完成，CCC=0.4985
Q7 加载完成，CCC=0.5171
Q8 加载完成，CCC=0.3374


In [107]:
restored_question_df = (
    pd.DataFrame(
        restored_question_rows
    )
)


display(
    restored_question_df.round(6)
)


print(
    "8题预测矩阵形状：",
    seed42_question_predictions.shape,
)

print(
    "8题概率张量形状：",
    seed42_question_probabilities.shape,
)

print(
    "最大CCC恢复差异：",
    restored_question_df[
        "CCC_Difference"
    ].max(),
)

,Question_Number,Question,Checkpoint_Epoch,Saved_CCC,Restored_CCC,CCC_Difference
0,1,PHQ_8NoInterest,9,0.453341,0.453341,0.0
1,2,PHQ_8Depressed,11,0.528571,0.528571,0.0
2,3,PHQ_8Sleep,15,0.415959,0.415959,0.0
3,4,PHQ_8Tired,10,0.471696,0.471696,0.0
4,5,PHQ_8Appetite,19,0.512979,0.512979,0.0
5,6,PHQ_8Failure,14,0.498478,0.498478,0.0
6,7,PHQ_8Concentrating,20,0.517125,0.517125,0.0
7,8,PHQ_8Moving,12,0.337409,0.337409,0.0


8题预测矩阵形状： torch.Size([55, 8])
8题概率张量形状： torch.Size([55, 8, 4])
最大CCC恢复差异： 0.0


In [108]:
def compute_total_score_metrics(
    predictions,
    labels,
):
    predictions = (
        predictions
        .detach()
        .cpu()
        .float()
        .reshape(-1)
    )

    labels = (
        labels
        .detach()
        .cpu()
        .float()
        .reshape(-1)
    )

    if predictions.shape != (
        labels.shape
    ):
        raise ValueError(
            "总分预测与标签形状不一致"
        )

    prediction_mean = (
        predictions.mean()
    )

    label_mean = (
        labels.mean()
    )

    prediction_variance = (
        predictions.var(
            correction=1
        )
    )

    label_variance = (
        labels.var(
            correction=1
        )
    )

    covariance = (
        (
            predictions
            - prediction_mean
        )
        * (
            labels
            - label_mean
        )
    ).sum() / max(
        len(labels) - 1,
        1,
    )

    ccc = (
        2.0 * covariance
        / (
            prediction_variance
            + label_variance
            + (
                prediction_mean
                - label_mean
            ).pow(2)
            + EPS
        )
    )

    rmse = torch.sqrt(
        F.mse_loss(
            predictions,
            labels,
        )
    )

    mae = F.l1_loss(
        predictions,
        labels,
    )

    exact_accuracy = (
        predictions
        == labels
    ).float().mean()

    return {
        "ccc": ccc.item(),
        "rmse": rmse.item(),
        "mae": mae.item(),
        "exact_accuracy": (
            exact_accuracy.item()
        ),
    }

In [109]:
seed42_predicted_total = (
    seed42_question_predictions
    .sum(dim=1)
)


seed42_true_total = (
    true_question_labels
    .sum(dim=1)
)


seed42_total_metrics = (
    compute_total_score_metrics(
        predictions=(
            seed42_predicted_total
        ),
        labels=(
            seed42_true_total
        ),
    )
)


print(
    "预测总分形状：",
    seed42_predicted_total.shape,
)

print(
    "预测总分范围：",
    (
        seed42_predicted_total
        .min()
        .item(),
        seed42_predicted_total
        .max()
        .item(),
    ),
)

print(
    "真实总分范围：",
    (
        seed42_true_total
        .min()
        .item(),
        seed42_true_total
        .max()
        .item(),
    ),
)


print()
print("Seed 42验证集总分结果")

print(
    "  CCC：",
    seed42_total_metrics["ccc"],
)

print(
    "  RMSE：",
    seed42_total_metrics["rmse"],
)

print(
    "  MAE：",
    seed42_total_metrics["mae"],
)

print(
    "  总分完全相等比例：",
    seed42_total_metrics[
        "exact_accuracy"
    ],
)

预测总分形状： torch.Size([55])
预测总分范围： (0, 21)
真实总分范围： (0, 20)

Seed 42验证集总分结果
  CCC： 0.6992999911308289
  RMSE： 4.242640495300293
  MAE： 3.200000047683716
  总分完全相等比例： 0.09090909361839294


In [110]:
seed42_prediction_df = (
    pd.DataFrame(
        {
            "Participant_ID": (
                val_dataset
                .participant_ids
            )
        }
    )
)


for question_index, question_name in enumerate(
    PHQ8_QUESTION_COLUMNS
):
    seed42_prediction_df[
        f"True_Q{question_index + 1}"
    ] = (
        true_question_labels[
            :,
            question_index,
        ].numpy()
    )

    seed42_prediction_df[
        f"Pred_Q{question_index + 1}"
    ] = (
        seed42_question_predictions[
            :,
            question_index,
        ].numpy()
    )


seed42_prediction_df[
    "True_Total"
] = seed42_true_total.numpy()


seed42_prediction_df[
    "Predicted_Total"
] = (
    seed42_predicted_total.numpy()
)


seed42_prediction_df[
    "Total_Error"
] = (
    seed42_prediction_df[
        "Predicted_Total"
    ]
    - seed42_prediction_df[
        "True_Total"
    ]
)


seed42_prediction_df[
    "Absolute_Total_Error"
] = (
    seed42_prediction_df[
        "Total_Error"
    ].abs()
)


seed42_prediction_path = (
    CHECKPOINT_ROOT
    / "seed_42_validation_predictions.csv"
)


seed42_prediction_df.to_csv(
    seed42_prediction_path,
    index=False,
)


display(
    seed42_prediction_df.head(
        10
    )
)


print(
    "预测结果已保存：",
    seed42_prediction_path,
)

,Participant_ID,True_Q1,Pred_Q1,True_Q2,Pred_Q2,True_Q3,Pred_Q3,True_Q4,Pred_Q4,True_Q5,...,True_Q6,Pred_Q6,True_Q7,Pred_Q7,True_Q8,Pred_Q8,True_Total,Predicted_Total,Total_Error,Absolute_Total_Error
0,300,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,2,0,-2,2
1,301,0,0,0,0,1,0,1,0,1,...,0,1,0,0,0,0,3,1,-2,2
2,306,0,1,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,2,2,2
3,317,1,1,1,2,1,1,1,2,1,...,1,2,2,2,0,0,8,11,3,3
4,320,1,1,1,2,3,3,1,2,2,...,1,1,1,2,1,2,11,15,4,4
5,321,2,2,3,2,3,3,3,3,3,...,3,3,3,3,0,2,20,21,1,1
6,331,1,2,1,2,1,1,1,2,1,...,1,3,1,3,1,0,8,14,6,6
7,334,1,0,1,0,1,0,1,0,0,...,1,1,0,0,0,0,5,1,-4,4
8,336,0,1,1,0,3,2,2,1,0,...,1,1,0,1,0,0,7,7,0,0
9,343,1,1,2,2,1,1,1,1,1,...,2,1,1,1,0,0,9,7,-2,2


预测结果已保存： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_42_validation_predictions.csv


In [112]:
def find_total_label_mismatches(
    metadata_df,
    split_name,
):
    item_total = torch.tensor(
        metadata_df[
            PHQ8_QUESTION_COLUMNS
        ].sum(axis=1).to_numpy(),
        dtype=torch.long,
    )

    official_total = torch.tensor(
        metadata_df[
            "PHQ_Score"
        ].to_numpy(),
        dtype=torch.long,
    )

    mismatch_positions = torch.where(
        item_total
        != official_total
    )[0].tolist()

    diagnostic_rows = []

    for position in mismatch_positions:
        participant_id = int(
            metadata_df.iloc[
                position
            ]["Participant_ID"]
        )

        detailed_row = (
            detailed_labels_df[
                detailed_labels_df[
                    "Participant_ID"
                ] == participant_id
            ]
        )

        if len(detailed_row) != 1:
            raise ValueError(
                f"参与者{participant_id}"
                "在详细标签表中不是唯一一行"
            )

        detailed_row = (
            detailed_row.iloc[0]
        )

        diagnostic_row = {
            "Split": split_name,
            "Position": position,
            "Participant_ID": (
                participant_id
            ),
            "Item_Sum": (
                item_total[
                    position
                ].item()
            ),
            "PHQ_Score": (
                official_total[
                    position
                ].item()
            ),
            "Difference": (
                item_total[
                    position
                ].item()
                - official_total[
                    position
                ].item()
            ),
        }

        if "PHQ_8Total" in (
            detailed_labels_df.columns
        ):
            diagnostic_row[
                "PHQ_8Total"
            ] = int(
                detailed_row[
                    "PHQ_8Total"
                ]
            )

        for question_index, question_name in enumerate(
            PHQ8_QUESTION_COLUMNS,
            start=1,
        ):
            diagnostic_row[
                f"Q{question_index}"
            ] = int(
                detailed_row[
                    question_name
                ]
            )

        diagnostic_rows.append(
            diagnostic_row
        )

    return pd.DataFrame(
        diagnostic_rows
    )

In [113]:
train_total_mismatch_df = (
    find_total_label_mismatches(
        metadata_df=(
            train_metadata_df
        ),
        split_name="train",
    )
)


val_total_mismatch_df = (
    find_total_label_mismatches(
        metadata_df=(
            val_metadata_df
        ),
        split_name="val",
    )
)


print(
    "训练集总分不一致人数：",
    len(
        train_total_mismatch_df
    ),
)

display(
    train_total_mismatch_df
)


print(
    "验证集总分不一致人数：",
    len(
        val_total_mismatch_df
    ),
)

display(
    val_total_mismatch_df
)

训练集总分不一致人数： 1


,Split,Position,Participant_ID,Item_Sum,PHQ_Score,Difference,PHQ_8Total,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8
0,train,136,485,4,2,2,4,1,1,0,1,0,1,0,0


验证集总分不一致人数： 1


,Split,Position,Participant_ID,Item_Sum,PHQ_Score,Difference,PHQ_8Total,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8
0,val,45,486,2,4,-2,2,0,1,0,0,0,0,0,1


In [114]:
seed42_detailed_total = (
    seed42_true_total.clone()
)


seed42_official_total = (
    torch.tensor(
        val_metadata_df[
            "PHQ_Score"
        ].to_numpy(),
        dtype=torch.long,
    )
)


seed42_official_metrics = (
    compute_total_score_metrics(
        predictions=(
            seed42_predicted_total
        ),
        labels=(
            seed42_official_total
        ),
    )
)


seed42_detailed_metrics = (
    compute_total_score_metrics(
        predictions=(
            seed42_predicted_total
        ),
        labels=(
            seed42_detailed_total
        ),
    )
)

In [115]:
seed42_label_source_comparison_df = (
    pd.DataFrame(
        [
            {
                "Label_Source": (
                    "Split_PHQ_Score"
                ),
                "CCC": (
                    seed42_official_metrics[
                        "ccc"
                    ]
                ),
                "RMSE": (
                    seed42_official_metrics[
                        "rmse"
                    ]
                ),
                "MAE": (
                    seed42_official_metrics[
                        "mae"
                    ]
                ),
                "Exact_Accuracy": (
                    seed42_official_metrics[
                        "exact_accuracy"
                    ]
                ),
            },
            {
                "Label_Source": (
                    "Detailed_Item_Sum"
                ),
                "CCC": (
                    seed42_detailed_metrics[
                        "ccc"
                    ]
                ),
                "RMSE": (
                    seed42_detailed_metrics[
                        "rmse"
                    ]
                ),
                "MAE": (
                    seed42_detailed_metrics[
                        "mae"
                    ]
                ),
                "Exact_Accuracy": (
                    seed42_detailed_metrics[
                        "exact_accuracy"
                    ]
                ),
            },
        ]
    )
)


display(
    seed42_label_source_comparison_df
    .round(6)
)

,Label_Source,CCC,RMSE,MAE,Exact_Accuracy
0,Split_PHQ_Score,0.694334,4.268276,3.236364,0.090909
1,Detailed_Item_Sum,0.699300,4.242640,3.200000,0.090909


In [116]:
seed42_prediction_df[
    "Detailed_Item_Total"
] = (
    seed42_detailed_total.numpy()
)


seed42_prediction_df[
    "Official_PHQ_Score"
] = (
    seed42_official_total.numpy()
)


seed42_prediction_df[
    "Total_Label_Discrepancy"
] = (
    seed42_prediction_df[
        "Detailed_Item_Total"
    ]
    - seed42_prediction_df[
        "Official_PHQ_Score"
    ]
)


# True_Total统一表示主要评价标签
seed42_prediction_df[
    "True_Total"
] = (
    seed42_official_total.numpy()
)


seed42_prediction_df[
    "Predicted_Total"
] = (
    seed42_predicted_total.numpy()
)


seed42_prediction_df[
    "Total_Error"
] = (
    seed42_prediction_df[
        "Predicted_Total"
    ]
    - seed42_prediction_df[
        "True_Total"
    ]
)


seed42_prediction_df[
    "Absolute_Total_Error"
] = (
    seed42_prediction_df[
        "Total_Error"
    ].abs()
)


seed42_prediction_df.to_csv(
    seed42_prediction_path,
    index=False,
)


display(
    seed42_prediction_df[
        seed42_prediction_df[
            "Total_Label_Discrepancy"
        ] != 0
    ]
)

,Participant_ID,True_Q1,Pred_Q1,True_Q2,Pred_Q2,True_Q3,Pred_Q3,True_Q4,Pred_Q4,True_Q5,...,Pred_Q7,True_Q8,Pred_Q8,True_Total,Predicted_Total,Total_Error,Absolute_Total_Error,Detailed_Item_Total,Official_PHQ_Score,Total_Label_Discrepancy
45,486,0,0,1,0,0,0,0,0,0,...,0,1,0,4,0,-4,4,2,4,-2


In [117]:
def build_total_score_result(
    seed,
    question_predictions,
    true_question_labels,
    participant_ids,
):
    """
    汇总某个随机种子的8道题预测，并计算PHQ-8总分指标。

    参数
    ----------
    seed:
        当前随机种子。

    question_predictions:
        8道题的预测结果，形状为 [55, 8]。

    true_question_labels:
        8道题的详细真实标签，形状为 [55, 8]。

    participant_ids:
        验证集参与者编号，长度为55。
    """

    # --------------------------------------------------
    # 1. 将8道题的预测分数相加
    # --------------------------------------------------
    predicted_total = question_predictions.sum(dim=1)

    # 形状变化：
    # [55, 8] --sum(dim=1)--> [55]
    #
    # dim=1表示沿着“8道题”这个维度求和，
    # 因此每个参与者最终只剩一个预测总分。


    # --------------------------------------------------
    # 2. 计算详细题目标签之和
    # --------------------------------------------------
    detailed_total = true_question_labels.sum(dim=1)

    # 这也是 [55]，
    # 但它来自 Detailed_PHQ8_Labels.csv 中8道题的真实分数。


    # --------------------------------------------------
    # 3. 读取split文件提供的官方总分
    # --------------------------------------------------
    official_total = torch.tensor(
        val_metadata_df["PHQ_Score"].to_numpy(),
        dtype=torch.long,
    )

    # to_numpy()：
    # 将Pandas的一列转换成NumPy数组。
    #
    # torch.tensor(...)：
    # 再把NumPy数组转换成PyTorch张量。
    #
    # dtype=torch.long：
    # PHQ分数是整数，所以使用int64类型。


    # --------------------------------------------------
    # 4. 分别计算两套指标
    # --------------------------------------------------
    official_metrics = compute_total_score_metrics(
        predicted_total,
        official_total,
    )

    detailed_metrics = compute_total_score_metrics(
        predicted_total,
        detailed_total,
    )


    # --------------------------------------------------
    # 5. 建立逐参与者结果表
    # --------------------------------------------------
    prediction_df = pd.DataFrame({
        "Participant_ID": participant_ids,
        "Predicted_Total": predicted_total.cpu().numpy(),
        "Official_PHQ_Score": official_total.cpu().numpy(),
        "Detailed_Item_Total": detailed_total.cpu().numpy(),
    })

    # 两种真实总分之间的差异
    prediction_df["Total_Label_Discrepancy"] = (
        prediction_df["Detailed_Item_Total"]
        - prediction_df["Official_PHQ_Score"]
    )

    # 预测误差统一以split中的PHQ_Score为准
    prediction_df["Total_Error"] = (
        prediction_df["Predicted_Total"]
        - prediction_df["Official_PHQ_Score"]
    )

    prediction_df["Absolute_Error"] = (
        prediction_df["Total_Error"].abs()
    )


    # --------------------------------------------------
    # 6. 汇总当前Seed的两套指标
    # --------------------------------------------------
    summary = {
        "Seed": seed,

        "Official_CCC": official_metrics["ccc"],
        "Official_RMSE": official_metrics["rmse"],
        "Official_MAE": official_metrics["mae"],
        "Official_Exact_Accuracy":
            official_metrics["exact_accuracy"],

        "Detailed_CCC": detailed_metrics["ccc"],
        "Detailed_RMSE": detailed_metrics["rmse"],
        "Detailed_MAE": detailed_metrics["mae"],
        "Detailed_Exact_Accuracy":
            detailed_metrics["exact_accuracy"],
    }

    return {
        "summary": summary,
        "prediction_df": prediction_df,
        "predicted_total": predicted_total,
        "official_total": official_total,
        "detailed_total": detailed_total,
        "official_metrics": official_metrics,
        "detailed_metrics": detailed_metrics,
    }

In [118]:
seed42_total_result = build_total_score_result(
    seed=42,
    question_predictions=seed42_question_predictions,
    true_question_labels=true_question_labels,
    participant_ids=val_metadata_df[
        "Participant_ID"
    ].tolist(),
)

In [119]:
seed42_total_summary_df = pd.DataFrame([
    seed42_total_result["summary"]
])

display(seed42_total_summary_df)

,Seed,Official_CCC,Official_RMSE,Official_MAE,Official_Exact_Accuracy,Detailed_CCC,Detailed_RMSE,Detailed_MAE,Detailed_Exact_Accuracy
0,42,0.694334,4.268276,3.236364,0.090909,0.6993,4.24264,3.2,0.090909


In [120]:
seed42_discrepancy_df = (
    seed42_total_result["prediction_df"]
    .query("Total_Label_Discrepancy != 0")
)

display(seed42_discrepancy_df)

,Participant_ID,Predicted_Total,Official_PHQ_Score,Detailed_Item_Total,Total_Label_Discrepancy,Total_Error,Absolute_Error
45,486,0,4,2,-2,-4,4


In [122]:
def train_all_questions_for_seed(
    seed,
):
    seed_results = {}
    summary_rows = []

    print()
    print("#" * 70)
    print(
        f"开始处理Seed {seed}的8道题"
    )
    print("#" * 70)

    for question_index in range(
        len(
            PHQ8_QUESTION_COLUMNS
        )
    ):
        result = train_or_load_question(
            question_index=(
                question_index
            ),
            seed=seed,
        )

        seed_results[
            question_index
        ] = result

        summary_rows.append(
            {
                "Question_Number": (
                    result[
                        "question_number"
                    ]
                ),
                "Question": (
                    result[
                        "question_name"
                    ]
                ),
                "Seed": seed,
                "Best_Loss": (
                    result[
                        "best_val_loss"
                    ]
                ),
                "Best_Loss_Epoch": (
                    result[
                        "best_loss_epoch"
                    ]
                ),
                "Best_CCC": (
                    result[
                        "best_val_ccc"
                    ]
                ),
                "Best_CCC_Epoch": (
                    result[
                        "best_ccc_epoch"
                    ]
                ),
            }
        )

    summary_df = pd.DataFrame(
        summary_rows
    )

    summary_path = (
        CHECKPOINT_ROOT
        / (
            f"seed_{seed}_"
            "question_summary.csv"
        )
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    print()
    print(
        f"Seed {seed}的8题训练完成"
    )

    print(
        "平均单题CCC：",
        summary_df[
            "Best_CCC"
        ].mean(),
    )

    return (
        seed_results,
        summary_df,
    )

In [134]:
def evaluate_seed_total_score(seed, seed_results):
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    question_count = len(PHQ8_QUESTION_COLUMNS)
    question_predictions = torch.empty((len(val_dataset), question_count), dtype=torch.long)
    per_question_rows = []

    for question_index in range(question_count):
        question_result = seed_results[question_index]
        checkpoint_path = question_result["best_ccc_path"]
        model, checkpoint = load_question_model(checkpoint_path=checkpoint_path, expected_question_index=question_index, expected_seed=seed)
        probabilities, predictions, labels = predict_single_question(model=model, dataloader=val_loader, question_index=question_index, device=device)
        expected_labels = val_dataset.labels[:, question_index]

        if not torch.equal(labels, expected_labels):
            raise RuntimeError("验证标签顺序不一致")

        question_predictions[:, question_index] = predictions
        restored_metrics = compute_metrics(predictions=predictions, labels=labels)
        saved_ccc = checkpoint["val_metrics"]["ccc"]

        question_row = {}
        question_row["Question_Number"] = question_index + 1
        question_row["Question"] = PHQ8_QUESTION_COLUMNS[question_index]
        question_row["Seed"] = seed
        question_row["Epoch"] = checkpoint["epoch"]
        question_row["CCC"] = restored_metrics["ccc"]
        question_row["Saved_CCC"] = saved_ccc
        question_row["CCC_Difference"] = abs(restored_metrics["ccc"] - saved_ccc)
        per_question_rows.append(question_row)

        del model
        del checkpoint
        del probabilities

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    true_question_labels = val_dataset.labels.clone()
    predicted_total = question_predictions.sum(dim=1)
    detailed_total = true_question_labels.sum(dim=1)

    metadata_participant_ids = torch.tensor(val_metadata_df["Participant_ID"].to_numpy(), dtype=torch.long)
    dataset_participant_ids = torch.tensor(val_dataset.participant_ids, dtype=torch.long)

    if not torch.equal(metadata_participant_ids, dataset_participant_ids):
        raise RuntimeError("val_metadata_df与val_dataset的参与者顺序不一致")

    official_total = torch.tensor(val_metadata_df["PHQ_Score"].to_numpy(), dtype=torch.long)

    official_metrics = compute_total_score_metrics(predictions=predicted_total, labels=official_total)
    detailed_metrics = compute_total_score_metrics(predictions=predicted_total, labels=detailed_total)

    prediction_df = pd.DataFrame({"Participant_ID": val_dataset.participant_ids})

    for question_index in range(question_count):
        prediction_df[f"True_Q{question_index + 1}"] = true_question_labels[:, question_index].numpy()
        prediction_df[f"Pred_Q{question_index + 1}"] = question_predictions[:, question_index].numpy()

    prediction_df["Official_PHQ_Score"] = official_total.numpy()
    prediction_df["Detailed_Item_Total"] = detailed_total.numpy()
    prediction_df["Predicted_Total"] = predicted_total.numpy()
    prediction_df["Total_Label_Discrepancy"] = prediction_df["Detailed_Item_Total"] - prediction_df["Official_PHQ_Score"]
    prediction_df["Total_Error"] = prediction_df["Predicted_Total"] - prediction_df["Official_PHQ_Score"]
    prediction_df["Absolute_Total_Error"] = prediction_df["Total_Error"].abs()

    prediction_path = CHECKPOINT_ROOT / f"seed_{seed}_validation_predictions.csv"
    prediction_df.to_csv(prediction_path, index=False)

    per_question_df = pd.DataFrame(per_question_rows)
    per_question_path = CHECKPOINT_ROOT / f"seed_{seed}_restored_question_metrics.csv"
    per_question_df.to_csv(per_question_path, index=False)

    result = {}
    result["seed"] = seed

    result["ccc"] = official_metrics["ccc"]
    result["rmse"] = official_metrics["rmse"]
    result["mae"] = official_metrics["mae"]
    result["exact_accuracy"] = official_metrics["exact_accuracy"]

    result["official_ccc"] = official_metrics["ccc"]
    result["official_rmse"] = official_metrics["rmse"]
    result["official_mae"] = official_metrics["mae"]
    result["official_exact_accuracy"] = official_metrics["exact_accuracy"]

    result["detailed_ccc"] = detailed_metrics["ccc"]
    result["detailed_rmse"] = detailed_metrics["rmse"]
    result["detailed_mae"] = detailed_metrics["mae"]
    result["detailed_exact_accuracy"] = detailed_metrics["exact_accuracy"]

    result["question_predictions"] = question_predictions
    result["true_question_labels"] = true_question_labels
    result["predicted_total"] = predicted_total
    result["official_total"] = official_total
    result["detailed_total"] = detailed_total
    result["true_total"] = official_total
    result["official_metrics"] = official_metrics
    result["detailed_metrics"] = detailed_metrics
    result["prediction_df"] = prediction_df
    result["per_question_df"] = per_question_df
    result["prediction_path"] = prediction_path
    result["per_question_path"] = per_question_path

    print()
    print("=" * 60)
    print(f"Seed {seed}总分验证结果")
    print("=" * 60)

    print("主要结果：Split文件中的PHQ_Score")
    print(f"  CCC：{result['official_ccc']:.6f}")
    print(f"  RMSE：{result['official_rmse']:.6f}")
    print(f"  MAE：{result['official_mae']:.6f}")
    print(f"  完全相等比例：{result['official_exact_accuracy']:.6f}")

    print()
    print("辅助结果：8道详细题目标签之和")
    print(f"  CCC：{result['detailed_ccc']:.6f}")
    print(f"  RMSE：{result['detailed_rmse']:.6f}")
    print(f"  MAE：{result['detailed_mae']:.6f}")
    print(f"  完全相等比例：{result['detailed_exact_accuracy']:.6f}")

    print()
    print(f"逐参与者结果已保存：{prediction_path}")
    print(f"逐题恢复结果已保存：{per_question_path}")

    return result

In [135]:
seed100_results, seed100_summary_df = (
    train_all_questions_for_seed(seed=100)
)

display(seed100_summary_df)


######################################################################
开始处理Seed 100的8道题
######################################################################
Q1 seed=100已经完成，直接读取已有结果
Q2 seed=100已经完成，直接读取已有结果
Q3 seed=100已经完成，直接读取已有结果
Q4 seed=100已经完成，直接读取已有结果
Q5 seed=100已经完成，直接读取已有结果
Q6 seed=100已经完成，直接读取已有结果
Q7 seed=100已经完成，直接读取已有结果
Q8 seed=100已经完成，直接读取已有结果

Seed 100的8题训练完成
平均单题CCC： 0.4220174290239811


,Question_Number,Question,Seed,Best_Loss,Best_Loss_Epoch,Best_CCC,Best_CCC_Epoch
0,1,PHQ_8NoInterest,100,2.432294,10,0.389335,13
1,2,PHQ_8Depressed,100,1.965447,8,0.485833,14
2,3,PHQ_8Sleep,100,2.297344,13,0.409947,13
3,4,PHQ_8Tired,100,2.060625,8,0.433423,17
4,5,PHQ_8Appetite,100,2.032872,11,0.462212,13
5,6,PHQ_8Failure,100,2.017258,13,0.446312,13
6,7,PHQ_8Concentrating,100,2.078422,12,0.491717,18
7,8,PHQ_8Moving,100,1.734236,12,0.257360,14


In [136]:
all_seed_question_results = {
    42: seed42_results
}


all_seed_question_summaries = {
    42: seed42_summary_df
}
for seed in [
    100,
    1234,
]:
    (
        seed_results,
        seed_summary_df,
    ) = train_all_questions_for_seed(
        seed=seed
    )

    all_seed_question_results[
        seed
    ] = seed_results

    all_seed_question_summaries[
        seed
    ] = seed_summary_df


######################################################################
开始处理Seed 100的8道题
######################################################################
Q1 seed=100已经完成，直接读取已有结果
Q2 seed=100已经完成，直接读取已有结果
Q3 seed=100已经完成，直接读取已有结果
Q4 seed=100已经完成，直接读取已有结果
Q5 seed=100已经完成，直接读取已有结果
Q6 seed=100已经完成，直接读取已有结果
Q7 seed=100已经完成，直接读取已有结果
Q8 seed=100已经完成，直接读取已有结果

Seed 100的8题训练完成
平均单题CCC： 0.4220174290239811

######################################################################
开始处理Seed 1234的8道题
######################################################################
Q1 seed=1234已经完成，直接读取已有结果
Q2 seed=1234已经完成，直接读取已有结果
Q3 seed=1234已经完成，直接读取已有结果
Q4 seed=1234已经完成，直接读取已有结果
Q5 seed=1234已经完成，直接读取已有结果
Q6 seed=1234已经完成，直接读取已有结果
Q7 seed=1234已经完成，直接读取已有结果
Q8 seed=1234已经完成，直接读取已有结果

Seed 1234的8题训练完成
平均单题CCC： 0.4312092401087284


In [137]:
all_seed_total_results = {}


for seed in EXPERIMENT_SEEDS:
    all_seed_total_results[
        seed
    ] = evaluate_seed_total_score(
        seed=seed,
        seed_results=(
            all_seed_question_results[
                seed
            ]
        ),
    )


Seed 42总分验证结果
主要结果：Split文件中的PHQ_Score
  CCC：0.694334
  RMSE：4.268276
  MAE：3.236364
  完全相等比例：0.090909

辅助结果：8道详细题目标签之和
  CCC：0.699300
  RMSE：4.242640
  MAE：3.200000
  完全相等比例：0.090909

逐参与者结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_42_validation_predictions.csv
逐题恢复结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_42_restored_question_metrics.csv

Seed 100总分验证结果
主要结果：Split文件中的PHQ_Score
  CCC：0.607187
  RMSE：4.992722
  MAE：3.545455
  完全相等比例：0.163636

辅助结果：8道详细题目标签之和
  CCC：0.612167
  RMSE：4.970824
  MAE：3.509091
  完全相等比例：0.163636

逐参与者结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_100_validation_predictions.csv
逐题恢复结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_100_restored_question_metrics.csv

Seed 1234总分验证结果
主要结果：Split文件中的PHQ_Score
  CCC：0.643476
  RMSE：4.558708
  MAE：3.509091
  完全相等比例：0.090909

辅助结果：8道详细题目标签之和
  CCC：0.646335
  RMSE：4.550725
  MAE：3.472727
  完全相等比例：0.109091

逐参与者结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_pape

In [138]:
seed_total_summary_rows = []


for seed in EXPERIMENT_SEEDS:
    result = (
        all_seed_total_results[
            seed
        ]
    )

    seed_total_summary_rows.append(
        {
            "Seed": seed,
            "CCC": result["ccc"],
            "RMSE": result["rmse"],
            "MAE": result["mae"],
            "Exact_Accuracy": (
                result[
                    "exact_accuracy"
                ]
            ),
        }
    )


seed_total_summary_df = (
    pd.DataFrame(
        seed_total_summary_rows
    )
)


display(
    seed_total_summary_df.round(4)
)

,Seed,CCC,RMSE,MAE,Exact_Accuracy
0,42,0.6943,4.2683,3.2364,0.0909
1,100,0.6072,4.9927,3.5455,0.1636
2,1234,0.6435,4.5587,3.5091,0.0909


In [139]:
three_seed_statistics_df = (
    pd.DataFrame(
        {
            "Metric": [
                "CCC",
                "RMSE",
                "MAE",
            ],
            "Mean": [
                seed_total_summary_df[
                    "CCC"
                ].mean(),

                seed_total_summary_df[
                    "RMSE"
                ].mean(),

                seed_total_summary_df[
                    "MAE"
                ].mean(),
            ],
            "Std": [
                seed_total_summary_df[
                    "CCC"
                ].std(ddof=1),

                seed_total_summary_df[
                    "RMSE"
                ].std(ddof=1),

                seed_total_summary_df[
                    "MAE"
                ].std(ddof=1),
            ],
        }
    )
)


statistics_path = (
    CHECKPOINT_ROOT
    / "three_seed_total_statistics.csv"
)


three_seed_statistics_df.to_csv(
    statistics_path,
    index=False,
)


display(
    three_seed_statistics_df.round(4)
)


print(
    "三种子统计已保存：",
    statistics_path,
)

,Metric,Mean,Std
0,CCC,0.6483,0.0438
1,RMSE,4.6066,0.3646
2,MAE,3.4303,0.1689


三种子统计已保存： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/three_seed_total_statistics.csv


In [141]:
corrected_seed42_total_result = evaluate_seed_total_score(seed=42, seed_results=seed42_results)


Seed 42总分验证结果
主要结果：Split文件中的PHQ_Score
  CCC：0.694334
  RMSE：4.268276
  MAE：3.236364
  完全相等比例：0.090909

辅助结果：8道详细题目标签之和
  CCC：0.699300
  RMSE：4.242640
  MAE：3.200000
  完全相等比例：0.090909

逐参与者结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_42_validation_predictions.csv
逐题恢复结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_42_restored_question_metrics.csv


In [143]:
seed1234_results, seed1234_summary_df = (
    train_all_questions_for_seed(seed=1234)
)


######################################################################
开始处理Seed 1234的8道题
######################################################################
Q1 seed=1234已经完成，直接读取已有结果
Q2 seed=1234已经完成，直接读取已有结果
Q3 seed=1234已经完成，直接读取已有结果
Q4 seed=1234已经完成，直接读取已有结果
Q5 seed=1234已经完成，直接读取已有结果
Q6 seed=1234已经完成，直接读取已有结果
Q7 seed=1234已经完成，直接读取已有结果
Q8 seed=1234已经完成，直接读取已有结果

Seed 1234的8题训练完成
平均单题CCC： 0.4312092401087284


In [144]:
corrected_seed100_total_result = evaluate_seed_total_score(seed=100, seed_results=seed100_results)
corrected_seed1234_total_result = evaluate_seed_total_score(seed=1234, seed_results=seed1234_results)


Seed 100总分验证结果
主要结果：Split文件中的PHQ_Score
  CCC：0.607187
  RMSE：4.992722
  MAE：3.545455
  完全相等比例：0.163636

辅助结果：8道详细题目标签之和
  CCC：0.612167
  RMSE：4.970824
  MAE：3.509091
  完全相等比例：0.163636

逐参与者结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_100_validation_predictions.csv
逐题恢复结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_100_restored_question_metrics.csv

Seed 1234总分验证结果
主要结果：Split文件中的PHQ_Score
  CCC：0.643476
  RMSE：4.558708
  MAE：3.509091
  完全相等比例：0.090909

辅助结果：8道详细题目标签之和
  CCC：0.646335
  RMSE：4.550725
  MAE：3.472727
  完全相等比例：0.109091

逐参与者结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_1234_validation_predictions.csv
逐题恢复结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_1234_restored_question_metrics.csv


In [145]:
corrected_total_results = [corrected_seed42_total_result, corrected_seed100_total_result, corrected_seed1234_total_result]
corrected_seed_rows = []

for result in corrected_total_results:
    row = {}
    row["Seed"] = result["seed"]
    row["Official_CCC"] = result["official_ccc"]
    row["Official_RMSE"] = result["official_rmse"]
    row["Official_MAE"] = result["official_mae"]
    row["Official_Exact_Accuracy"] = result["official_exact_accuracy"]
    row["Detailed_CCC"] = result["detailed_ccc"]
    row["Detailed_RMSE"] = result["detailed_rmse"]
    row["Detailed_MAE"] = result["detailed_mae"]
    row["Detailed_Exact_Accuracy"] = result["detailed_exact_accuracy"]
    corrected_seed_rows.append(row)

corrected_seed_summary_df = pd.DataFrame(corrected_seed_rows)
display(corrected_seed_summary_df)

,Seed,Official_CCC,Official_RMSE,Official_MAE,Official_Exact_Accuracy,Detailed_CCC,Detailed_RMSE,Detailed_MAE,Detailed_Exact_Accuracy
0,42,0.694334,4.268276,3.236364,0.090909,0.699300,4.242640,3.200000,0.090909
1,100,0.607187,4.992722,3.545455,0.163636,0.612167,4.970824,3.509091,0.163636
2,1234,0.643476,4.558708,3.509091,0.090909,0.646335,4.550725,3.472727,0.109091


In [146]:
official_summary_rows = []

for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    column_name = f"Official_{metric_name}"
    row = {}
    row["Metric"] = metric_name
    row["Mean"] = corrected_seed_summary_df[column_name].mean()
    row["Std"] = corrected_seed_summary_df[column_name].std(ddof=1)
    official_summary_rows.append(row)

official_three_seed_summary_df = pd.DataFrame(official_summary_rows)
display(official_three_seed_summary_df)

,Metric,Mean,Std
0,CCC,0.648332,0.043776
1,RMSE,4.606569,0.364587
2,MAE,3.430303,0.168938
3,Exact_Accuracy,0.115152,0.041989


In [147]:
corrected_seed_summary_path = CHECKPOINT_ROOT / "three_seed_total_results.csv"
official_summary_path = CHECKPOINT_ROOT / "three_seed_official_summary.csv"
corrected_seed_summary_df.to_csv(corrected_seed_summary_path, index=False)
official_three_seed_summary_df.to_csv(official_summary_path, index=False)
print(f"逐Seed结果：{corrected_seed_summary_path}")
print(f"三Seed正式汇总：{official_summary_path}")

逐Seed结果：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/three_seed_total_results.csv
三Seed正式汇总：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/three_seed_official_summary.csv


In [148]:
corrected_seed_summary_path = CHECKPOINT_ROOT / "three_seed_total_results.csv"
official_summary_path = CHECKPOINT_ROOT / "three_seed_official_summary.csv"
corrected_seed_summary_df.to_csv(corrected_seed_summary_path, index=False)
official_three_seed_summary_df.to_csv(official_summary_path, index=False)
print(f"逐Seed完整结果：{corrected_seed_summary_path}")
print(f"正式均值和标准差：{official_summary_path}")
print(f"逐Seed文件存在：{corrected_seed_summary_path.exists()}")
print(f"正式汇总文件存在：{official_summary_path.exists()}")

逐Seed完整结果：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/three_seed_total_results.csv
正式均值和标准差：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/three_seed_official_summary.csv
逐Seed文件存在：True
正式汇总文件存在：True


In [149]:
TEST_SPLIT_PATH = SPLIT_DIR / "test_split.csv"
TEST_CSV_PATH = SPLIT_DIR / "test.csv"

test_split_df = pd.read_csv(TEST_SPLIT_PATH)
test_csv_df = pd.read_csv(TEST_CSV_PATH)

print("test_split.csv形状：", test_split_df.shape)
print("test_split.csv列名：", test_split_df.columns.tolist())
print("test.csv形状：", test_csv_df.shape)
print("test.csv列名：", test_csv_df.columns.tolist())

display(test_split_df.head())
display(test_csv_df.head())

test_split.csv形状： (56, 6)
test_split.csv列名： ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']
test.csv形状： (56, 6)
test.csv列名： ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']


,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity
0,600,female,0,5,0,23.0
1,602,female,1,13,1,67.0
2,604,male,1,12,0,30.0
3,605,male,0,2,0,23.0
4,606,female,0,5,0,46.0


,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity
0,600,female,0,5,0,23.0
1,602,female,1,13,1,67.0
2,604,male,1,12,0,30.0
3,605,male,0,2,0,23.0
4,606,female,0,5,0,46.0


In [150]:
print("test_split.csv缺失值数量：")
print(test_split_df.isna().sum())

print()
print("test.csv缺失值数量：")
print(test_csv_df.isna().sum())

test_split.csv缺失值数量：
Participant_ID    0
Gender            0
PHQ_Binary        0
PHQ_Score         0
PCL-C (PTSD)      0
PTSD Severity     2
dtype: int64

test.csv缺失值数量：
Participant_ID    0
Gender            0
PHQ_Binary        0
PHQ_Score         0
PCL-C (PTSD)      0
PTSD Severity     2
dtype: int64


In [151]:
missing_test_transcripts = []
existing_test_transcripts = []

for participant_id in test_split_df["Participant_ID"].astype(int).tolist():
    transcript_path = DATA_DIR / f"{participant_id}_P" / f"{participant_id}_Transcript.csv"

    if transcript_path.exists():
        existing_test_transcripts.append(transcript_path)
    else:
        missing_test_transcripts.append(transcript_path)

print("存在的测试转录数量：", len(existing_test_transcripts))
print("缺失的测试转录数量：", len(missing_test_transcripts))
print("缺失文件：", missing_test_transcripts)

存在的测试转录数量： 56
缺失的测试转录数量： 0
缺失文件： []


In [153]:
import inspect
train_participant_ids = set(train_metadata_df["Participant_ID"].astype(int).tolist())
val_participant_ids = set(val_metadata_df["Participant_ID"].astype(int).tolist())
test_participant_ids = set(test_split_df["Participant_ID"].astype(int).tolist())

print("test_split形状：", test_split_df.shape)
print("test.csv与test_split.csv完全相同：", test_csv_df.equals(test_split_df))
print("训练与测试重叠：", train_participant_ids & test_participant_ids)
print("验证与测试重叠：", val_participant_ids & test_participant_ids)
print("get_participant_embeddings参数：", inspect.signature(get_participant_embeddings))
print("compute_participant_embeddings参数：", inspect.signature(compute_participant_embeddings))
print("CachedPHQ8Dataset参数：", inspect.signature(CachedPHQ8Dataset))

test_split形状： (56, 6)
test.csv与test_split.csv完全相同： True
训练与测试重叠： set()
验证与测试重叠： set()
get_participant_embeddings参数： (participant_id)
compute_participant_embeddings参数： (participant_id)
CachedPHQ8Dataset参数： (metadata_df, split_name)


In [154]:
test_example_participant_id = int(test_split_df.iloc[0]["Participant_ID"])
test_example_embedding, test_example_mask, test_example_turn_count, test_example_cache_hit = get_participant_embeddings(test_example_participant_id)

print("测试参与者：", test_example_participant_id)
print("句向量形状：", test_example_embedding.shape)
print("Mask形状：", test_example_mask.shape)
print("真实对话轮数：", test_example_turn_count)
print("Padding轮数：", test_example_mask.sum().item())
print("是否命中缓存：", test_example_cache_hit)
print("句向量设备：", test_example_embedding.device)
print("句向量类型：", test_example_embedding.dtype)
print("Mask类型：", test_example_mask.dtype)

测试参与者： 600
句向量形状： torch.Size([120, 768])
Mask形状： torch.Size([120])
真实对话轮数： 70
Padding轮数： 50
是否命中缓存： False
句向量设备： cpu
句向量类型： torch.float32
Mask类型： torch.bool


In [155]:
class CachedTestDataset(Dataset):
    def __init__(self, metadata_df):
        super().__init__()
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.total_scores = torch.tensor(self.metadata_df["PHQ_Score"].to_numpy(), dtype=torch.long)
        embedding_list = []
        mask_list = []
        turn_count_list = []
        cache_hit_count = 0

        for index, participant_id in enumerate(self.participant_ids):
            embedding, mask, turn_count, cache_hit = get_participant_embeddings(participant_id)
            embedding_list.append(embedding)
            mask_list.append(mask)
            turn_count_list.append(turn_count)

            if cache_hit:
                cache_hit_count += 1

            if (index + 1) % 10 == 0 or index + 1 == len(self.participant_ids):
                print(f"测试集句向量加载进度：{index + 1}/{len(self.participant_ids)}")

        self.embeddings = torch.stack(embedding_list, dim=0)
        self.masks = torch.stack(mask_list, dim=0)
        self.turn_counts = torch.tensor(turn_count_list, dtype=torch.long)
        self.cache_hit_count = cache_hit_count
        self.cache_miss_count = len(self.participant_ids) - cache_hit_count

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        embedding = self.embeddings[index]
        mask = self.masks[index]
        total_score = self.total_scores[index]
        participant_id = self.participant_ids[index]
        return embedding, mask, total_score, participant_id

In [157]:
test_dataset = CachedTestDataset(test_split_df)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
test_batch_embeddings, test_batch_masks, test_batch_scores, test_batch_participant_ids = next(iter(test_loader))

print("测试样本数：", len(test_dataset))
print("测试批次数：", len(test_loader))
print("批次句向量：", test_batch_embeddings.shape)
print("批次Mask：", test_batch_masks.shape)
print("批次真实总分：", test_batch_scores.shape)
print("批次参与者编号：", test_batch_participant_ids.shape)
print("第一批参与者：", test_batch_participant_ids)
print("第一批真实总分：", test_batch_scores)
print("缓存命中数量：", test_dataset.cache_hit_count)
print("缓存未命中数量：", test_dataset.cache_miss_count)

测试集句向量加载进度：10/56
测试集句向量加载进度：20/56
测试集句向量加载进度：30/56
测试集句向量加载进度：40/56
测试集句向量加载进度：50/56
测试集句向量加载进度：56/56
测试样本数： 56
测试批次数： 6
批次句向量： torch.Size([10, 120, 768])
批次Mask： torch.Size([10, 120])
批次真实总分： torch.Size([10])
批次参与者编号： torch.Size([10])
第一批参与者： tensor([600, 602, 604, 605, 606, 607, 609, 615, 618, 619])
第一批真实总分： tensor([ 5, 13, 12,  2,  5,  7,  0,  3,  4,  6])
缓存命中数量： 56
缓存未命中数量： 0


In [161]:
def evaluate_seed_on_test(seed, seed_results, test_loader):
    question_count = len(PHQ8_QUESTION_COLUMNS)
    test_sample_count = len(test_loader.dataset)
    participant_ids = torch.tensor(test_loader.dataset.participant_ids, dtype=torch.long)
    true_total = test_loader.dataset.total_scores.clone()
    question_predictions = torch.empty((test_sample_count, question_count), dtype=torch.long)
    selected_checkpoint_rows = []

    for question_index in range(question_count):
        question_result = seed_results[question_index]
        checkpoint_path = question_result["best_ccc_path"]
        model, checkpoint = load_question_model(checkpoint_path=checkpoint_path, expected_question_index=question_index, expected_seed=seed)
        model.eval()
        batch_prediction_list = []
        batch_participant_id_list = []

        with torch.no_grad():
            for batch_embeddings, batch_masks, batch_scores, batch_participant_ids in test_loader:
                batch_embeddings = batch_embeddings.to(device)
                batch_masks = batch_masks.to(device)
                logits = model(batch_embeddings, batch_masks)
                batch_predictions = torch.argmax(logits, dim=1)
                batch_prediction_list.append(batch_predictions.cpu())
                batch_participant_id_list.append(batch_participant_ids.cpu())

        question_prediction = torch.cat(batch_prediction_list, dim=0)
        observed_participant_ids = torch.cat(batch_participant_id_list, dim=0)

        if not torch.equal(observed_participant_ids, participant_ids):
            raise RuntimeError("测试集参与者顺序不一致")

        if question_prediction.shape[0] != test_sample_count:
            raise RuntimeError("测试集预测数量不正确")

        question_predictions[:, question_index] = question_prediction

        checkpoint_row = {}
        checkpoint_row["Question_Number"] = question_index + 1
        checkpoint_row["Question"] = PHQ8_QUESTION_COLUMNS[question_index]
        checkpoint_row["Seed"] = seed
        checkpoint_row["Selected_Epoch"] = checkpoint["epoch"]
        checkpoint_row["Validation_CCC"] = checkpoint["val_metrics"]["ccc"]
        checkpoint_row["Checkpoint_Path"] = str(checkpoint_path)
        selected_checkpoint_rows.append(checkpoint_row)

        print(f"Seed {seed}，题目{question_index + 1}/8测试完成")

        del model
        del checkpoint
        del logits

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    predicted_total = question_predictions.sum(dim=1)
    test_metrics = compute_total_score_metrics(predictions=predicted_total, labels=true_total)

    prediction_df = pd.DataFrame({"Participant_ID": participant_ids.numpy()})

    for question_index in range(question_count):
        prediction_df[f"Pred_Q{question_index + 1}"] = question_predictions[:, question_index].numpy()

    prediction_df["True_Total"] = true_total.numpy()
    prediction_df["Predicted_Total"] = predicted_total.numpy()
    prediction_df["Total_Error"] = prediction_df["Predicted_Total"] - prediction_df["True_Total"]
    prediction_df["Absolute_Total_Error"] = prediction_df["Total_Error"].abs()

    prediction_path = CHECKPOINT_ROOT / f"seed_{seed}_test_predictions_56.csv"
    prediction_df.to_csv(prediction_path, index=False)

    selected_checkpoint_df = pd.DataFrame(selected_checkpoint_rows)
    selected_checkpoint_path = CHECKPOINT_ROOT / f"seed_{seed}_test_selected_checkpoints.csv"
    selected_checkpoint_df.to_csv(selected_checkpoint_path, index=False)

    result = {}
    result["seed"] = seed
    result["sample_count"] = test_sample_count
    result["ccc"] = test_metrics["ccc"]
    result["rmse"] = test_metrics["rmse"]
    result["mae"] = test_metrics["mae"]
    result["exact_accuracy"] = test_metrics["exact_accuracy"]
    result["question_predictions"] = question_predictions
    result["predicted_total"] = predicted_total
    result["true_total"] = true_total
    result["participant_ids"] = participant_ids
    result["prediction_df"] = prediction_df
    result["selected_checkpoint_df"] = selected_checkpoint_df
    result["prediction_path"] = prediction_path
    result["selected_checkpoint_path"] = selected_checkpoint_path

    print()
    print("=" * 60)
    print(f"Seed {seed}独立测试集结果")
    print("=" * 60)
    print(f"测试参与者数量：{test_sample_count}")
    print(f"CCC：{result['ccc']:.6f}")
    print(f"RMSE：{result['rmse']:.6f}")
    print(f"MAE：{result['mae']:.6f}")
    print(f"完全相等比例：{result['exact_accuracy']:.6f}")
    print(f"预测总分范围：{predicted_total.min().item()}～{predicted_total.max().item()}")
    print(f"真实总分范围：{true_total.min().item()}～{true_total.max().item()}")
    print(f"预测结果已保存：{prediction_path}")

    return result

In [162]:
seed42_test_result = evaluate_seed_on_test(seed=42, seed_results=seed42_results, test_loader=test_loader)

Seed 42，题目1/8测试完成
Seed 42，题目2/8测试完成
Seed 42，题目3/8测试完成
Seed 42，题目4/8测试完成
Seed 42，题目5/8测试完成
Seed 42，题目6/8测试完成
Seed 42，题目7/8测试完成
Seed 42，题目8/8测试完成

Seed 42独立测试集结果
测试参与者数量：56
CCC：0.562100
RMSE：6.040045
MAE：4.589286
完全相等比例：0.071429
预测总分范围：0～22
真实总分范围：0～22
预测结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_42_test_predictions_56.csv


In [163]:
seed100_test_result = evaluate_seed_on_test(seed=100, seed_results=seed100_results, test_loader=test_loader)
seed1234_test_result = evaluate_seed_on_test(seed=1234, seed_results=seed1234_results, test_loader=test_loader)


Seed 100，题目1/8测试完成
Seed 100，题目2/8测试完成
Seed 100，题目3/8测试完成
Seed 100，题目4/8测试完成
Seed 100，题目5/8测试完成
Seed 100，题目6/8测试完成
Seed 100，题目7/8测试完成
Seed 100，题目8/8测试完成

Seed 100独立测试集结果
测试参与者数量：56
CCC：0.597337
RMSE：5.890368
MAE：4.625000
完全相等比例：0.107143
预测总分范围：0～22
真实总分范围：0～22
预测结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_100_test_predictions_56.csv
Seed 1234，题目1/8测试完成
Seed 1234，题目2/8测试完成
Seed 1234，题目3/8测试完成
Seed 1234，题目4/8测试完成
Seed 1234，题目5/8测试完成
Seed 1234，题目6/8测试完成
Seed 1234，题目7/8测试完成
Seed 1234，题目8/8测试完成

Seed 1234独立测试集结果
测试参与者数量：56
CCC：0.658989
RMSE：5.294877
MAE：4.071429
完全相等比例：0.071429
预测总分范围：0～22
真实总分范围：0～22
预测结果已保存：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/seed_1234_test_predictions_56.csv


In [164]:
all_test_results = [seed42_test_result, seed100_test_result, seed1234_test_result]
test_seed_rows = []

for result in all_test_results:
    row = {}
    row["Seed"] = result["seed"]
    row["Sample_Count"] = result["sample_count"]
    row["CCC"] = result["ccc"]
    row["RMSE"] = result["rmse"]
    row["MAE"] = result["mae"]
    row["Exact_Accuracy"] = result["exact_accuracy"]
    test_seed_rows.append(row)

test_seed_summary_df = pd.DataFrame(test_seed_rows)
display(test_seed_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,56,0.562100,6.040045,4.589286,0.071429
1,100,56,0.597337,5.890368,4.625000,0.107143
2,1234,56,0.658989,5.294877,4.071429,0.071429


In [165]:
test_summary_rows = []

for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    row = {}
    row["Metric"] = metric_name
    row["Mean"] = test_seed_summary_df[metric_name].mean()
    row["Std"] = test_seed_summary_df[metric_name].std(ddof=1)
    test_summary_rows.append(row)

test_three_seed_summary_df = pd.DataFrame(test_summary_rows)
display(test_three_seed_summary_df)

,Metric,Mean,Std
0,CCC,0.606142,0.049041
1,RMSE,5.741763,0.394185
2,MAE,4.428572,0.309810
3,Exact_Accuracy,0.083333,0.020620


CCC:0.615+- 0.031
RMSE: 5.71+-0.25
MAE:4.36+-0.26

In [166]:
test_seed_summary_path = CHECKPOINT_ROOT / "three_seed_test_results_56.csv"
test_three_seed_summary_path = CHECKPOINT_ROOT / "three_seed_test_summary_56.csv"
test_seed_summary_df.to_csv(test_seed_summary_path, index=False)
test_three_seed_summary_df.to_csv(test_three_seed_summary_path, index=False)
print(f"逐Seed测试结果：{test_seed_summary_path}")
print(f"三Seed测试汇总：{test_three_seed_summary_path}")

逐Seed测试结果：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/three_seed_test_results_56.csv
三Seed测试汇总：/workspace/E-DAIC/checkpoints/phq8_text_paperlike/three_seed_test_summary_56.csv
